In [ ]:
# %% [Cell 0] Environment
import sys
print(sys.executable)


In [ ]:
# %% [Cell 1] Imports, configuration, helpers, structure loading
# ------------------------------------------------------------------
# Periodic sets use CARTESIAN coordinates (r = f·C). The submitted version
# paired fractional coordinates with an Å cell (reviewer comment, Fig. 3a).
# pymatgen ideal-fraction rounding is disabled (FRAC_TOLERANCE = 0) so that
# coordinates are used exactly as published; this makes the pipeline
# identical to amd.CifReader. Parser warnings are logged, not suppressed.
# ------------------------------------------------------------------

import re
import json
import pickle
import inspect
import datetime
import platform
import warnings
from collections import Counter
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

from pymatgen.core import Structure
from pymatgen.io.cif import CifParser
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import amd
from amd import AMD, PDD, PeriodicSet
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

plt.rcParams["figure.dpi"] = 300
plt.rcParams["font.family"] = "Arial"
np.random.seed(0)

# ── CONFIGURATION ────────────────────────────────────────────────────────────
K_MIN      = 1
K_MAX      = 100
K_BASELINE = 5          # baseline for AMD and clustering (Å) — state in Methods

COLLAPSE_TOL   = 1e-4          # PDD row-merging tolerance (Å) — state in Methods
FRAC_TOLERANCE = 0.0           # pymatgen ideal-fraction rounding OFF (pymatgen default is 1e-4)
ALT_TOL        = 1e-4 if FRAC_TOLERANCE == 0 else 0.0   # the setting NOT used (sensitivity tests)

AMD_METRIC     = "euclidean"   # L∞: package default for AMD_pdist and the EMD ground metric
AMD_PDIST_METRIC = "euclidean"   # distance between AMD vectors (Ward assumes Euclidean); EMD stays L∞
LINKAGE_METHOD = "ward"        # NB: Ward formally assumes Euclidean distances (see notes)

MODEL_MIN_SITES = 40           # P1 cells with ≥ this many sites are treated as periodic models
K_TSNE          = None         # None → min k_reliable over periodic models (capped at K_MAX)
MIN_NN_DIST_A   = 1.2          # sanity floor for nearest-neighbour distance in ZnO (Å)

SRC_COLORS = {"MP": "#1565C0", "COD": "#B71C1C", "OQMD": "#2E7D32", "MATTERGEN": "#6A1B9A", "other": "#757575"}

cif_dir          = Path(r"c:\Users\browlins\OneDrive - University of Edinburgh\Documents\Python_Scripts\amd\cif")
RESULTS_BASE_DIR = cif_dir.parent / "results"
SUBFOLDERS       = ["test", "mp", "cod", "oqmd", "mg"]

PLOT_K_SWEEP_SUMMARY = True   # True → generate the four-panel summary figure for every k in K_SWEEP

if "frac_tolerance" not in inspect.signature(CifParser.__init__).parameters:
    raise RuntimeError("Installed pymatgen has no CifParser(frac_tolerance=...) option — please upgrade pymatgen.")

# ── RUN DIRECTORY ────────────────────────────────────────────────────────────
run_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = RESULTS_BASE_DIR / f"run_{run_timestamp}"
run_dir.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {run_dir}")

# ── GENERAL HELPERS ──────────────────────────────────────────────────────────
def get_source(name):
    if name.startswith("mp-"):   return "MP"
    if name.startswith("cod-"):  return "COD"
    if name.startswith("oqmd-"): return "OQMD"
    if name.startswith("mg-"):   return "MATTERGEN"
    return "other"

def short_label(name):
    return name.replace(".cif", "")

def _pkg_version(dist):
    try:
        return version(dist)
    except PackageNotFoundError:
        return "unknown"

# ── PERIODIC SETS AND DESCRIPTORS ────────────────────────────────────────────
def _check_cartesian(motif, cell, name=""):
    """Guard against the fractional-coordinate error found in review: for cells
    with all axes > 1.5 Å, a Cartesian motif cannot lie wholly inside [0, 1)^3."""
    if np.linalg.norm(cell, axis=1).min() > 1.5 and np.all((motif > -1e-6) & (motif < 1.0)):
        raise ValueError(f"{name}: motif lies wholly in [0,1)^3 — fractional coordinates passed as Cartesian?")

def structure_to_periodicset(structure, name=""):
    """pymatgen Structure → amd PeriodicSet with a CARTESIAN motif (Å)."""
    motif = np.asarray(structure.cart_coords, dtype=float)      # r = f · C
    cell  = np.asarray(structure.lattice.matrix, dtype=float)
    _check_cartesian(motif, cell, name)
    return PeriodicSet(motif, cell)

def legacy_fractional_periodicset(structure):
    """ORIGINAL (FAULTY) conversion: fractional motif + Å cell.
    Kept ONLY to document the artefact (Cells 4, 5, 11). DO NOT USE FOR ANALYSIS."""
    return PeriodicSet(structure.frac_coords, structure.lattice.matrix)

def compute_pdd(ps, k=K_MAX, collapse=True):
    """PDD of a PeriodicSet. Column 0 = row weights; columns 1..k = distances (Å)."""
    if not isinstance(ps, PeriodicSet):
        raise TypeError(f"compute_pdd expects a PeriodicSet, got {type(ps).__name__}; "
                        f"use structure_pdd() for pymatgen Structures")
    return np.asarray(PDD(ps, k, collapse=collapse, collapse_tol=COLLAPSE_TOL), dtype=float)

def structure_pdd(structure, k=K_MAX, name="", collapse=True):
    """PDD of a pymatgen Structure via the Cartesian conversion."""
    return compute_pdd(structure_to_periodicset(structure, name), k, collapse=collapse)

def emd(D1, D2, k=None):
    """EMD between PDDs (L∞ ground metric), optionally truncated to the first k neighbours."""
    if k is None:
        k = min(D1.shape[1], D2.shape[1]) - 1
    return float(amd.EMD(np.ascontiguousarray(D1[:, :k + 1]),
                         np.ascontiguousarray(D2[:, :k + 1]), metric=AMD_METRIC))

def pairwise_emd(names, pdds, k=None):
    """Symmetric EMD matrix (plain loop: avoids multiprocessing issues on Windows/Jupyter)."""
    n = len(names)
    M = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            M[i, j] = M[j, i] = emd(pdds[names[i]], pdds[names[j]], k)
    return M

def inscribed_radius(lattice):
    """Half the smallest interplanar spacing of the cell faces (Å)."""
    a, b, c = lattice.matrix
    V = abs(np.linalg.det(lattice.matrix))
    d = [V / np.linalg.norm(np.cross(b, c)),
         V / np.linalg.norm(np.cross(c, a)),
         V / np.linalg.norm(np.cross(a, b))]
    return 0.5 * min(d)

def k_reliable_for(structure):
    """Approximate number of neighbours inside the inscribed sphere."""
    rho = structure.num_sites / structure.lattice.volume
    r   = inscribed_radius(structure.lattice)
    return max(1, int(np.floor(4.0 / 3.0 * np.pi * r**3 * rho)))

# ── CIF PARSING ──────────────────────────────────────────────────────────────
_ROUNDED = re.compile(r"(\d+)\s+fractional coordinates?\s+rounded", re.IGNORECASE)

def rounded_count(msgs):
    """Number of coordinates pymatgen reports rounding to ideal fractions."""
    for m in msgs:
        hit = _ROUNDED.search(m)
        if hit:
            return int(hit.group(1))
    return 0

def parse_cif(path, frac_tolerance=FRAC_TOLERANCE):
    """Parse a CIF with explicit control of ideal-fraction rounding.
    Returns (first Structure, sorted list of parser warnings). Raises on failure."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        parser = CifParser(str(path), frac_tolerance=frac_tolerance)
        try:
            structures = parser.parse_structures(primitive=False)
        except AttributeError:                           # pymatgen < 2024.2
            structures = parser.get_structures(primitive=False)
    msgs = [str(w.message) for w in caught] + [str(m) for m in getattr(parser, "warnings", [])]
    return structures[0], sorted(set(msgs))

parse_log = {}   # file name → list of parser warnings

def load_structure(path):
    try:
        s, msgs = parse_cif(path, FRAC_TOLERANCE)
        parse_log[path.name] = msgs
        return s
    except Exception as e:
        parse_log[path.name] = [f"LOAD FAILED: {type(e).__name__}: {e}"]
        print(f"  Error loading {path.name}: {e}")
        return None

def cifreader_pdd(path, k=K_MAX):
    """Independent cross-check: PDD from the package's own CIF reader. Returns (PDD or None, note)."""
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            ps = next(iter(amd.CifReader(str(path))), None)
    except Exception as e:
        return None, f"CifReader failed: {type(e).__name__}: {e}"
    if ps is None:
        return None, "CifReader yielded no structure"
    return compute_pdd(ps, k), "; ".join(sorted({str(w.message)[:120] for w in caught}))

# ── LOAD CIF FILES ───────────────────────────────────────────────────────────
all_structures, cif_paths = {}, {}
for sub in SUBFOLDERS:
    folder = cif_dir / sub
    if not folder.exists():
        continue
    loaded = 0
    for path in sorted(folder.glob("*.cif")):
        struct = load_structure(path)
        if struct is not None and struct.composition.reduced_formula == "ZnO":
            if path.name in all_structures:
                print(f"  WARNING: duplicate file name {path.name} in '{sub}' overrides earlier entry")
            all_structures[path.name] = struct
            cif_paths[path.name] = path
            loaded += 1
    print(f"  {sub:6s}: {loaded} ZnO structures")

print(f"\nTotal: {len(all_structures)} ZnO structures")
for src in ["MP", "COD", "OQMD", "MATTERGEN", "other"]:
    n = sum(1 for nm in all_structures if get_source(nm) == src)
    if n:
        print(f"  {src}: {n}")

# ── PARSER-WARNING LOG ───────────────────────────────────────────────────────
with open(run_dir / "parse_warnings.json", "w") as fh:
    json.dump(parse_log, fh, indent=2)

n_warn_files = sum(1 for v in parse_log.values() if v)
print(f"\npymatgen parser warnings for {n_warn_files} / {len(parse_log)} files → parse_warnings.json")
for msg, c in Counter(m[:100] for v in parse_log.values() for m in v).most_common(8):
    print(f"  [{c:3d}×] {msg}")

rounded_files = {f: rounded_count(m) for f, m in parse_log.items() if rounded_count(m)}
if FRAC_TOLERANCE == 0:
    print("  Ideal-fraction rounding: disabled" +
          (f" — BUT reported for {rounded_files}" if rounded_files else ", none applied ✓"))
else:
    print(f"  Ideal-fraction rounding applied in {len(rounded_files)} files: {rounded_files}")

# ── PROVENANCE ───────────────────────────────────────────────────────────────
run_info = {
    "timestamp": run_timestamp,
    "python": platform.python_version(),
    "average-minimum-distance": _pkg_version("average-minimum-distance"),
    "pymatgen": _pkg_version("pymatgen"),
    "numpy": np.__version__,
    "K_MAX": K_MAX, "K_BASELINE": K_BASELINE,
    "COLLAPSE_TOL": COLLAPSE_TOL, "FRAC_TOLERANCE": FRAC_TOLERANCE, "ALT_TOL": ALT_TOL,
    "AMD_METRIC": AMD_METRIC, "LINKAGE_METHOD": LINKAGE_METHOD,
    "MODEL_MIN_SITES": MODEL_MIN_SITES, "MIN_NN_DIST_A": MIN_NN_DIST_A,
    "cif_parser": "pymatgen CifParser.parse_structures(primitive=False), frac_tolerance=FRAC_TOLERANCE",
    "coordinate_handling": "Cartesian motif from pymatgen cart_coords (r = f·C); "
                           "cross-validated against amd.CifReader (Cell 4, test A6a)",
    "n_structures": len(all_structures),
    "n_files_with_parser_warnings": n_warn_files,
}
with open(run_dir / "run_info.json", "w") as fh:
    json.dump(run_info, fh, indent=2)


In [ ]:
# %% [Cell 2] CIF metadata (one row per CIF)
rows, model_flags, k_rel = [], {}, {}

for name, struct in sorted(all_structures.items()):
    comp, lat = struct.composition, struct.lattice
    try:
        sga       = SpacegroupAnalyzer(struct)
        sg_sym    = sga.get_space_group_symbol()
        sg_num    = sga.get_space_group_number()
        cry_sys   = sga.get_crystal_system()
        n_sym_ops = len(sga.get_symmetry_operations())
    except Exception as e:
        print(f"  SpacegroupAnalyzer failed for {name}: {e}")
        sg_sym, sg_num, cry_sys, n_sym_ops = "?", -1, "?", 0

    model_flags[name] = (sg_num == 1) and (struct.num_sites >= MODEL_MIN_SITES)
    k_rel[name]       = k_reliable_for(struct)

    el_frac = {str(el): float(comp.get_atomic_fraction(el))
               for el in sorted(comp.elements, key=lambda e: e.Z)}

    rows.append({
        "file_name":          name,
        "source":             get_source(name),
        "formula":            comp.formula,
        "reduced_formula":    comp.reduced_formula,
        "space_group_symbol": sg_sym,
        "space_group_number": sg_num,
        "n_sites":            struct.num_sites,
        "Z_formula_units":    int(round(comp.get_reduced_composition_and_factor()[1])),
        "density_g_cm3":      round(struct.density, 4),
        "volume_A3":          round(lat.volume, 4),
        "volume_per_atom_A3": round(lat.volume / struct.num_sites, 4),
        "a_A": round(lat.a, 4), "b_A": round(lat.b, 4), "c_A": round(lat.c, 4),
        "alpha_deg": round(lat.alpha, 4), "beta_deg": round(lat.beta, 4), "gamma_deg": round(lat.gamma, 4),
        "crystal_system":     cry_sys,
        "n_sym_ops":          n_sym_ops,
        "is_ordered":         struct.is_ordered,
        "is_periodic_model":  model_flags[name],
        "r_inscribed_A":      round(inscribed_radius(lat), 4),
        "k_reliable":         k_rel[name],
        "n_coords_rounded":   rounded_count(parse_log.get(name, [])),
        **el_frac,
    })

meta_df = pd.DataFrame(rows)
meta_df.to_csv(run_dir / "cif_metadata.csv", index=False)
print(f"Saved {len(meta_df)} rows → cif_metadata.csv")
print(f"Periodic models (P1, ≥{MODEL_MIN_SITES} sites): "
      f"{[(short_label(n), k_rel[n]) for n, f in model_flags.items() if f]}  (name, k_reliable)")
meta_df.head()


In [ ]:
# %% [Cell 3] PDD (k = K_MAX) and AMD (computed once; AMD(k) = first k entries)
pdd_store, amd_full, pdd_rows, amd_rows = {}, {}, [], []

for name in sorted(all_structures):
    try:
        ps = structure_to_periodicset(all_structures[name], name)
        D  = compute_pdd(ps, K_MAX)
        A  = np.asarray(AMD(ps, K_MAX), dtype=float)
    except Exception as e:
        print(f"  WARNING descriptors({name}): {type(e).__name__}: {e}")
        continue

    A_from_D = np.average(D[:, 1:], weights=D[:, 0], axis=0)     # AMD = weighted mean of PDD rows
    if not np.allclose(A, A_from_D, atol=10 * COLLAPSE_TOL):
        raise RuntimeError(f"{name}: AMD ≠ weighted mean of PDD rows "
                           f"(max dev {np.abs(A - A_from_D).max():.2e} Å)")

    pdd_store[name], amd_full[name] = D, A
    for r_idx in range(D.shape[0]):
        r = {"name": name, "row": r_idx, "weight": float(D[r_idx, 0])}
        r.update({f"d{i+1}": float(D[r_idx, i + 1]) for i in range(K_MAX)})
        pdd_rows.append(r)
    amd_rows.append({"name": name, **{f"AMD_{i+1}": float(v) for i, v in enumerate(A)}})

valid_names   = sorted(pdd_store)
amd_desc_by_k = {k: {n: amd_full[n][:k] for n in valid_names} for k in range(K_MIN, K_MAX + 1)}

with open(run_dir / "pdd_store.pkl", "wb") as fh:
    pickle.dump(pdd_store, fh)
pd.DataFrame(pdd_rows).to_csv(run_dir / "pdd_store.csv", index=False)
pd.DataFrame(amd_rows).to_csv(run_dir / "amd_descriptors.csv", index=False)
print(f"PDD/AMD computed for {len(valid_names)} / {len(all_structures)} structures (k ≤ {K_MAX})")


In [ ]:
# %% [Cell 4] Descriptor audit and representation-invariance tests
# ------------------------------------------------------------------
#   A  per-structure sanity checks + independent parser       (pass/fail; A6b info)
#   B  closed-form nearest-neighbour distances                (pass/fail)
#   C  representation invariance                              (pass/fail; C4 info)
#   D  flattened-PDD behaviour (submitted t-SNE input)        (info)
#   E  size of the submitted fractional-coordinate artefact   (info)
# EMD between PDDs is non-decreasing in k (L∞ ground metric), so EMD ≤ tol
# at K_MAX implies EMD ≤ tol for every k ≤ K_MAX.
# Results are saved BEFORE any AssertionError is raised.
# ------------------------------------------------------------------

TOL_WEIGHTS    = 1e-9    # |Σw − 1|
TOL_MONOTONE   = 1e-9    # Å, allowed decrease along a PDD row (float noise)
TOL_PARSER     = 1e-5    # Å, pipeline vs amd.CifReader
TOL_ANALYTIC   = 1e-4    # Å, limited by 4-decimal lattice parameters
TOL_INVARIANCE = 1e-5    # Å, change of cell / basis
PERTURB_A      = 5e-4    # Å, displacement used in D2/D3 (> COLLAPSE_TOL, so merged rows split)

SUPERCELL_SMALL = [2, 2, 2]                          # n_sites ≤ 24
SUPERCELL_LARGE = [2, 1, 1]                          # larger cells (runtime)
SHEAR_BASIS     = [[1, 1, 0], [0, 1, 0], [0, 0, 1]]  # unimodular, det = 1

TESTS = {   # id: (label, criterion, kind)
    "A1":  ("A1 motif Cartesian: max|x| (Å)",                           "motif not wholly in [0,1)^3", "check"),
    "A2":  ("A2 |sum(weights) - 1|",                                     f"<= {TOL_WEIGHTS:g}",         "check"),
    "A3":  ("A3 all PDD entries finite",                                 "1 = all finite",              "check"),
    "A4":  ("A4 nearest-neighbour distance (Å) — quote MIN",             f">= {MIN_NN_DIST_A:g} Å",     "check"),
    "A5":  ("A5 PDD rows non-decreasing in k",                           "1 = monotone",                "check"),
    "A6a": ("A6a EMD(pipeline, amd.CifReader) (Å)",                      f"<= {TOL_PARSER:g} Å",        "check"),
    "A6b": (f"A6b EMD(frac_tolerance {FRAC_TOLERANCE:g} vs {ALT_TOL:g}) (Å)", "informational",          "info"),
    "B1":  ("B1 |AMD_1 - analytic| (Å)",                                 f"<= {TOL_ANALYTIC:g} Å",      "check"),
    "C1":  ("C1 EMD(as-loaded, supercell) (Å)",                          f"<= {TOL_INVARIANCE:g} Å",    "check"),
    "C2":  ("C2 EMD(as-loaded, sheared basis) (Å)",                      f"<= {TOL_INVARIANCE:g} Å",    "check"),
    "C3":  ("C3 EMD(std primitive, std conventional) (Å)",               f"<= {TOL_INVARIANCE:g} Å",    "check"),
    "C4":  ("C4 EMD(as-loaded, std conventional) (Å) — incl. symmetrisation", "informational",          "info"),
    "D1":  ("D1 ||flat(PDD prim) - flat(PDD 2x2x2)|| (padded)",          "informational",               "info"),
    "D2":  (f"D2 ||flat(PDD) - flat(PDD perturbed {PERTURB_A:g} Å)|| (padded)", "informational",        "info"),
    "D3":  (f"D3 EMD(PDD, PDD perturbed {PERTURB_A:g} Å) (Å)",           "informational",               "info"),
    "E1":  ("E1 EMD(legacy fractional, corrected) (Å)",                  "informational",               "info"),
    "E2":  ("E2 legacy max d_(n-1) (fractional units)",                  "informational",               "info"),
    "E3":  ("E3 legacy min d_n (Å-scale lattice step)",                  "informational",               "info"),
}

records = []
def rec(test, structure, value, passed=None, note=""):
    records.append({"test": test, "structure": short_label(structure),
                    "value": np.nan if value is None else float(value),
                    "passed": None if passed is None else bool(passed), "note": note})

# ── Per-structure tests ──────────────────────────────────────────────────────
print(f"Auditing {len(valid_names)} structures at k = {K_MAX} …")
for i_s, name in enumerate(valid_names, start=1):
    s, D = all_structures[name], pdd_store[name]
    n = s.num_sites

    # A1–A5: well-formedness
    motif = np.asarray(s.cart_coords, dtype=float)
    rec("A1", name, np.abs(motif).max(), not np.all((motif > -1e-6) & (motif < 1.0)))
    w_err = abs(D[:, 0].sum() - 1.0)
    rec("A2", name, w_err, w_err <= TOL_WEIGHTS)
    finite = bool(np.isfinite(D).all())
    rec("A3", name, float(finite), finite)
    rec("A4", name, D[:, 1].min(), D[:, 1].min() >= MIN_NN_DIST_A, "self-distance excluded by construction")
    mono = bool((np.diff(D[:, 1:], axis=1) >= -TOL_MONOTONE).all())
    rec("A5", name, float(mono), mono)

    # A6a: independent parser
    D_cr, note_cr = cifreader_pdd(cif_paths[name], K_MAX)
    if D_cr is None:
        rec("A6a", name, None, False, note_cr)
    else:
        e = emd(D, D_cr, K_MAX)
        rec("A6a", name, e, e <= TOL_PARSER, note_cr)

    # A6b: effect of pymatgen ideal-fraction rounding
    try:
        s_alt, _ = parse_cif(cif_paths[name], ALT_TOL)
        rec("A6b", name, emd(D, structure_pdd(s_alt, K_MAX, f"{name}/alt"), K_MAX))
    except Exception as ex:
        rec("A6b", name, None, None, f"{type(ex).__name__}: {ex}")

    # C1: supercell
    mult = SUPERCELL_SMALL if n <= 24 else SUPERCELL_LARGE
    try:
        sc = s.copy(); sc.make_supercell(mult)
        e = emd(D, structure_pdd(sc, K_MAX, f"{name}/supercell"), K_MAX)
        rec("C1", name, e, e <= TOL_INVARIANCE, "x".join(map(str, mult)))
    except Exception as ex:
        rec("C1", name, None, False, f"{type(ex).__name__}: {ex}")

    # C2: unimodular change of basis
    try:
        sh = s.copy(); sh.make_supercell(SHEAR_BASIS)
        e = emd(D, structure_pdd(sh, K_MAX, f"{name}/shear"), K_MAX)
        rec("C2", name, e, e <= TOL_INVARIANCE)
    except Exception as ex:
        rec("C2", name, None, False, f"{type(ex).__name__}: {ex}")

    # C3 / C4: standard primitive vs standard conventional; as-loaded vs conventional
    try:
        sga    = SpacegroupAnalyzer(s)
        D_prim = structure_pdd(sga.get_primitive_standard_structure(),    K_MAX, f"{name}/prim")
        D_conv = structure_pdd(sga.get_conventional_standard_structure(), K_MAX, f"{name}/conv")
        e3 = emd(D_prim, D_conv, K_MAX)
        rec("C3", name, e3, e3 <= TOL_INVARIANCE)
        rec("C4", name, emd(D, D_conv, K_MAX), None, "includes SpacegroupAnalyzer symmetrisation (symprec 0.01 Å)")
    except Exception as ex:
        rec("C3", name, None, False, f"{type(ex).__name__}: {ex}")
        rec("C4", name, None, None, f"{type(ex).__name__}: {ex}")

    # E1–E3: the submitted artefact
    try:
        D_leg = compute_pdd(legacy_fractional_periodicset(s), max(K_MAX, n), collapse=False)
        rec("E1", name, emd(D_leg, D, K_MAX))
        if n >= 2:
            rec("E2", name, D_leg[:, n - 1].max())
            rec("E3", name, D_leg[:, n].min())
    except Exception as ex:
        rec("E1", name, None, None, f"{type(ex).__name__}: {ex}")

    if i_s % 10 == 0 or i_s == len(valid_names):
        print(f"  {i_s}/{len(valid_names)}")

# ── B1: closed-form AMD_1 (conventional cubic a, independent of cell setting) ─
ANALYTIC_AMD1 = {
    "mp-2229.cif":  ("rock salt: a/2",      lambda a: a / 2),
    "mp-1986.cif":  ("zinc blende: √3·a/4", lambda a: np.sqrt(3) * a / 4),
    "mp-13161.cif": ("CsCl type: √3·a/2",   lambda a: np.sqrt(3) * a / 2),
}
an_rows = []
for name, (desc, f) in ANALYTIC_AMD1.items():
    if name not in pdd_store:
        print(f"  (B1: {name} not in dataset — skipped)")
        continue
    try:
        sga = SpacegroupAnalyzer(all_structures[name])
        if sga.get_crystal_system() != "cubic":
            raise ValueError(f"expected cubic, got {sga.get_crystal_system()}")
        a_conv = sga.get_conventional_standard_structure().lattice.a
        amd1, exact = float(amd_full[name][0]), float(f(a_conv))
        rec("B1", name, abs(amd1 - exact), abs(amd1 - exact) <= TOL_ANALYTIC, desc)
        an_rows.append({"structure": short_label(name), "formula": desc, "a_conv_A": a_conv,
                        "AMD_1_A": amd1, "analytic_A": exact, "abs_diff_A": abs(amd1 - exact)})
    except Exception as ex:
        rec("B1", name, None, False, f"{type(ex).__name__}: {ex}")
pd.DataFrame(an_rows).to_csv(run_dir / "analytic_checks.csv", index=False)

# ── D1–D3: behaviour of the flattened, zero-padded PDD (submitted t-SNE input) ─
def _flat_pair(Pa, Pb):
    L = max(Pa.size, Pb.size)
    return np.pad(Pa.ravel(), (0, L - Pa.size)), np.pad(Pb.ravel(), (0, L - Pb.size))

D1_ID = next((n for n in ["mp-2229.cif", "mp-1986.cif"] if n in pdd_store), valid_names[0])
try:
    prim = SpacegroupAnalyzer(all_structures[D1_ID]).get_primitive_standard_structure()
    sc   = prim.copy(); sc.make_supercell([2, 2, 2])
    Pa, Pb = structure_pdd(prim, K_MAX, f"{D1_ID}/prim"), structure_pdd(sc, K_MAX, f"{D1_ID}/2x2x2")
    fa, fb = _flat_pair(Pa, Pb)
    rec("D1", D1_ID, np.linalg.norm(fa - fb), None, f"rows {Pa.shape[0]} vs {Pb.shape[0]} (merged + lexsorted)")
except Exception as ex:
    rec("D1", D1_ID, None, None, f"{type(ex).__name__}: {ex}")

D2_ID = next((n for n in ["mp-2133.cif", "mp-2229.cif"] if n in pdd_store), valid_names[0])
try:
    s0  = all_structures[D2_ID]
    rng = np.random.default_rng(0)
    v   = rng.normal(size=(s0.num_sites, 3))
    v  *= PERTURB_A / np.linalg.norm(v, axis=1, keepdims=True)
    s_p = Structure(s0.lattice, s0.species, s0.cart_coords + v, coords_are_cartesian=True)
    Pa, Pb = pdd_store[D2_ID], structure_pdd(s_p, K_MAX, f"{D2_ID}/perturbed")
    fa, fb = _flat_pair(Pa, Pb)
    note = f"rows {Pa.shape[0]} → {Pb.shape[0]}; every atom moved {PERTURB_A:g} Å"
    rec("D2", D2_ID, np.linalg.norm(fa - fb), None, note)
    rec("D3", D2_ID, emd(Pa, Pb, K_MAX), None, note)
except Exception as ex:
    rec("D2", D2_ID, None, None, f"{type(ex).__name__}: {ex}")

# ── Global pairwise EMD matrix at K_MAX ──────────────────────────────────────
emd_all = pairwise_emd(valid_names, pdd_store, K_MAX)
_lbl = [short_label(n) for n in valid_names]
pd.DataFrame(emd_all, index=_lbl, columns=_lbl).to_csv(run_dir / f"pdd_emd_matrix_k{K_MAX:03d}.csv")
i_max, j_max = np.unravel_index(np.argmax(emd_all), emd_all.shape)
global_max_emd = float(emd_all.max())

# ── Summary ──────────────────────────────────────────────────────────────────
rec_df = pd.DataFrame(records)
rec_df.to_csv(run_dir / "audit_records.csv", index=False)
rec_df.pivot_table(index="structure", columns="test", values="value", aggfunc="first") \
      .to_csv(run_dir / "audit_per_structure.csv")

summary_rows = []
for tid, (label, crit, kind) in TESTS.items():
    sub  = rec_df[rec_df["test"] == tid]
    vals = sub["value"].dropna()
    chk  = kind == "check"
    summary_rows.append({
        "id": tid, "test": label, "kind": kind, "criterion": crit, "n": len(sub),
        "n_passed":  int(sub["passed"].eq(True).sum())  if chk else pd.NA,
        "n_failed":  int(sub["passed"].eq(False).sum()) if chk else pd.NA,
        "n_nonzero": int((vals > 0).sum()),
        "min_value": vals.min() if len(vals) else np.nan,
        "max_value": vals.max() if len(vals) else np.nan,
    })
summary_df = pd.DataFrame(summary_rows)
summary_df[["n_passed", "n_failed"]] = summary_df[["n_passed", "n_failed"]].astype("Int64")
summary_df.to_csv(run_dir / "audit_summary.csv", index=False)

failures = rec_df[rec_df["test"].map(lambda t: TESTS[t][2] == "check") & rec_df["passed"].eq(False)]
failures.to_csv(run_dir / "audit_failures.csv", index=False)

with pd.option_context("display.width", 250, "display.max_colwidth", 75,
                       "display.float_format", "{:.3e}".format):
    print(summary_df.drop(columns=["criterion"]).to_string(index=False))
    if len(failures):
        print("\nFailures:")
        print(failures.to_string(index=False))

print(f"\nGlobal max EMD (k={K_MAX}): {global_max_emd:.3f} Å  "
      f"[{_lbl[i_max]} ↔ {_lbl[j_max]}]")

if len(failures):
    raise AssertionError(f"Audit FAILED: {len(failures)} check(s) — see audit_failures.csv")
print("AUDIT PASSED — all pass/fail checks within tolerance.")


In [ ]:
# %% [Cell 5] Reproduction of the submitted fractional-coordinate artefact (Fig. S[N])
ARTEFACT_DEMO = "mp-1245015.cif" if "mp-1245015.cif" in pdd_store else \
                next((n for n in valid_names if model_flags.get(n)), valid_names[0])
s_demo   = all_structures[ARTEFACT_DEMO]
n_demo   = s_demo.num_sites
D_legacy = compute_pdd(legacy_fractional_periodicset(s_demo), max(K_MAX, n_demo), collapse=False)
D_fixed  = pdd_store[ARTEFACT_DEMO]
k_ax     = np.arange(1, K_MAX + 1)

ks = [k for k in range(n_demo - 3, n_demo + 4) if 1 <= k <= K_MAX]
print(f"{short_label(ARTEFACT_DEMO)} (n_sites = {n_demo}), row 0, k = {ks[0]}…{ks[-1]}")
print("  faulty   :", np.round(D_legacy[0, ks], 3))
print("  corrected:", np.round(D_fixed[0, ks], 3))

fig_s, ax_s = plt.subplots(figsize=(8, 5))
for r in D_legacy:
    ax_s.plot(k_ax, r[1:K_MAX + 1], color="#B71C1C", lw=0.4, alpha=0.4)
for r in D_fixed:
    ax_s.plot(k_ax, r[1:], color="#1565C0", lw=0.4, alpha=0.4)
if n_demo <= K_MAX:
    ax_s.axvline(n_demo, color="k", ls="--", lw=1)
    ax_s.text(n_demo + 1, 0.5, f"k = n$_{{sites}}$ = {n_demo}", fontsize=10)
ax_s.legend(handles=[Line2D([0], [0], color="#B71C1C", lw=2, label="Submitted: fractional motif + Å cell (faulty)"),
                     Line2D([0], [0], color="#1565C0", lw=2, label="Revised: Cartesian motif (corrected)")],
            loc="upper left", fontsize=10)
ax_s.set_xlabel("Neighbour Index (k)")
ax_s.set_ylabel("Neighbour Distance [Å]")
ax_s.set_title(short_label(ARTEFACT_DEMO), fontsize=11)
ax_s.grid(True, ls="--", alpha=0.5)
fig_s.savefig(run_dir / "figS_fractional_artefact.png", dpi=300, bbox_inches="tight")
fig_s.savefig(run_dir / "figS_fractional_artefact.pdf", bbox_inches="tight")
plt.show()

# Where does the faulty AMD jump for each structure? (expected: k = n_sites)
jump_rows = []
for name in valid_names:
    s = all_structures[name]
    if s.num_sites >= K_MAX:
        continue
    d = np.diff(np.asarray(AMD(legacy_fractional_periodicset(s), K_MAX), dtype=float))  # d[i] = A[i+2] − A[i+1]
    jump_rows.append({"name": short_label(name), "n_sites": s.num_sites,
                      "faulty_jump_at_k": int(np.argmax(d)) + 2, "jump_size_A": float(d.max())})
jump_df = pd.DataFrame(jump_rows)
jump_df.to_csv(run_dir / "legacy_jump_table.csv", index=False)
n_jump_match = int((jump_df["faulty_jump_at_k"] == jump_df["n_sites"]).sum())
print(f"Faulty AMD jumps at k = n_sites for {n_jump_match} / {len(jump_df)} structures")


In [ ]:
# %% [Cell 6] pymatgen ideal-fraction rounding: diagnosis and effect on every result
# Compares the pipeline (FRAC_TOLERANCE) with the alternative setting (ALT_TOL).
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import fcluster
from sklearn.metrics import adjusted_rand_score

def _max_site_shift(s_a, s_b):
    """Largest periodic distance (Å) from a site in s_a to the nearest same-species site in s_b."""
    worst = 0.0
    for sp in sorted({site.species_string for site in s_a}):
        fa = np.array([site.frac_coords for site in s_a if site.species_string == sp])
        fb = np.array([site.frac_coords for site in s_b if site.species_string == sp])
        if len(fa) and len(fb):
            worst = max(worst, float(s_a.lattice.get_all_distances(fa, fb).min(axis=1).max()))
    return worst

rounding_in_alt = ALT_TOL > FRAC_TOLERANCE
pdd_alt, amd_alt, diag_rows = {}, {}, []

for name in valid_names:
    s0 = all_structures[name]
    s1, msgs1 = parse_cif(cif_paths[name], ALT_TOL)
    ps1 = structure_to_periodicset(s1, f"{name}/alt")
    pdd_alt[name] = compute_pdd(ps1, K_MAX)
    amd_alt[name] = np.asarray(AMD(ps1, K_MAX), dtype=float)
    same_n = s0.num_sites == s1.num_sites
    diag_rows.append({
        "structure":        short_label(name),
        "n_coords_rounded": rounded_count(msgs1 if rounding_in_alt else parse_log.get(name, [])),
        "max_atom_shift_A": _max_site_shift(s0, s1) if same_n else np.nan,
        "EMD_pipeline_vs_alt_A": emd(pdd_store[name], pdd_alt[name], K_MAX),
    })
    if model_flags.get(name) and same_n:     # list exact changes for P1 models
        dF = s1.frac_coords - s0.frac_coords
        dF -= np.round(dF)
        for a, c in np.argwhere(np.abs(dF) > 1e-12):
            print(f"  {short_label(name)} site {a} ({s0[a].species_string}), {'xyz'[c]}: "
                  f"{s0.frac_coords[a, c] % 1:.8f} (pipeline) vs {s1.frac_coords[a, c] % 1:.8f} (alt)")

diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(run_dir / "rounding_diagnosis.csv", index=False)
affected = diag_df[(diag_df["n_coords_rounded"] > 0) | (diag_df["EMD_pipeline_vs_alt_A"] > 0)]
with pd.option_context("display.float_format", "{:.2e}".format, "display.width", 200):
    print(f"Structures affected by rounding: {len(affected)} / {len(diag_df)}")
    print(affected.to_string(index=False))

# ── Sensitivity of results ───────────────────────────────────────────────────
iu    = np.triu_indices(len(valid_names), 1)
e_vec = diag_df["EMD_pipeline_vs_alt_A"].to_numpy()
M0    = emd_all
M1    = pairwise_emd(valid_names, pdd_alt, K_MAX)
bound = e_vec[:, None] + e_vec[None, :]               # triangle-inequality bound per pair
n_close = int(((M0 <= bound) & (bound > 0))[iu].sum())

X0 = np.array([amd_full[n][:K_BASELINE] for n in valid_names])
X1 = np.array([amd_alt[n][:K_BASELINE]  for n in valid_names])
d0 = amd.AMD_pdist(X0, metric=AMD_PDIST_METRIC) # AMD_METRIC
d1 = amd.AMD_pdist(X1, metric=AMD_PDIST_METRIC) # AMD_METRIC
Z0 = linkage(d0, method=LINKAGE_METHOD, optimal_ordering=True)
Z1 = linkage(d1, method=LINKAGE_METHOD, optimal_ordering=True)
leaves0 = dendrogram(Z0, no_plot=True)["leaves"]
leaves1 = dendrogram(Z1, no_plot=True)["leaves"]
c0 = fcluster(Z0, 0.65 * Z0[:, 2].max(), criterion="distance")
c1 = fcluster(Z1, 0.65 * Z1[:, 2].max(), criterion="distance")

sensitivity = {
    "n_structures_affected":        int(len(affected)),
    "max_EMD_pipeline_vs_alt_A":    float(e_vec.max()),
    "pdd_emd_max_abs_change_A":     float(np.abs(M0 - M1).max()),
    "pdd_emd_spearman":             float(spearmanr(M0[iu], M1[iu]).correlation),
    "smallest_pairwise_emd_A":      [float(x) for x in np.sort(M0[iu])[:8]],
    "pairs_within_rounding_bound":  n_close,
    "amd_max_abs_change_A":         float(np.abs(d0 - d1).max()),
    "amd_spearman":                 float(spearmanr(d0, d1).correlation),
    "dendrogram_leaf_order_identical": bool(leaves0 == leaves1),
    "flat_clusters_ARI":            float(adjusted_rand_score(c0, c1)),
}
print(json.dumps(sensitivity, indent=2))

if leaves0 != leaves1:
    diff = [(p, _lbl[a], _lbl[b]) for p, (a, b) in enumerate(zip(leaves0, leaves1)) if a != b]
    print("Leaf positions that differ (position, pipeline, alternative):", diff)

if "cod-2300112.cif" in valid_names:            # highlighted structure: AMD neighbours, both settings
    i = valid_names.index("cod-2300112.cif")
    for lbl_, dd in [(f"frac_tol={FRAC_TOLERANCE:g}", d0), (f"frac_tol={ALT_TOL:g}", d1)]:
        row = squareform(dd)[i]
        nn  = np.argsort(row)[1:6]
        print(f"  cod-2300112 nearest (AMD, {lbl_}):", [(_lbl[j], round(row[j], 5)) for j in nn])

with open(run_dir / "rounding_sensitivity.json", "w") as fh:
    json.dump(sensitivity, fh, indent=2)


In [ ]:
# %% [Cell 7] t-SNE embedding of the PDD EMD matrix (isometry-invariant input)
# The submitted version embedded zero-padded, flattened PDDs. These are not a valid
# comparison between distributions with different numbers of rows, and are
# discontinuous under small displacements (Cell 4, D2 vs D3). The EMD is a
# continuous metric on PDDs. k is capped at the smallest k_reliable among
# periodic models unless K_TSNE is set.
from sklearn.manifold import TSNE

model_names = [n for n in valid_names if model_flags.get(n, False)]
if K_TSNE is not None:
    K_TSNE_EFF = int(min(K_TSNE, K_MAX))
elif model_names:
    K_TSNE_EFF = int(min([K_MAX] + [k_rel[n] for n in model_names]))
else:
    K_TSNE_EFF = K_MAX
print(f"t-SNE on PDD EMDs at k = {K_TSNE_EFF}  "
      f"(periodic models: {[(short_label(n), k_rel[n]) for n in model_names]})")

emd_tsne = pairwise_emd(valid_names, pdd_store, K_TSNE_EFF)
pd.DataFrame(emd_tsne, index=_lbl, columns=_lbl).to_csv(run_dir / f"pdd_emd_matrix_k{K_TSNE_EFF:03d}.csv")

TSNE_PERPLEXITY_EFF = max(2.0, min(30.0, (len(valid_names) - 1) / 3.0))
_tsne_kw = dict(n_components=2, metric="precomputed", init="random",   # 'pca' init not allowed with precomputed
                perplexity=TSNE_PERPLEXITY_EFF, learning_rate="auto", random_state=42)
_tsne_kw["max_iter" if "max_iter" in inspect.signature(TSNE.__init__).parameters else "n_iter"] = 2000
tsne   = TSNE(**_tsne_kw)
coords = tsne.fit_transform(emd_tsne)
print(f"t-SNE done ({len(valid_names)} structures, perplexity {TSNE_PERPLEXITY_EFF:.1f}), "
      f"KL divergence = {tsne.kl_divergence_:.4f}")

pd.DataFrame({"name": _lbl, "tsne_1": coords[:, 0], "tsne_2": coords[:, 1]}).to_csv(
    run_dir / "tsne_coords.csv", index=False)


In [ ]:
if PLOT_K_SWEEP_SUMMARY:
    # %% [Cell 8] Summary figure (four panels) — sweep over k
    # ------------------------------------------------------------------
    # One four-panel figure per k in K_SWEEP = K_MIN, 5, 10, …, K_MAX:
    #  (a) PDD rows vs neighbour index (all k ≤ K_MAX); dashed line marks the current k
    #  (b) t-SNE of the PDD EMD matrix at k_t = min(k, K_TSNE_EFF)   [TSNE_FOLLOWS_K = True]
    #      or at K_TSNE_EFF in every frame                          [TSNE_FOLLOWS_K = False]
    #  (c) AMD dendrogram (AMD_pdist, L∞, first k AMD values)
    #  (d) AMD distance heatmap, lower triangle, dendrogram leaf order
    # Frames → run_dir/k_sweep/. The K_BASELINE frame is also saved to run_dir
    # (PNG + PDF) and shown inline. Afterwards Z_amd, order, mat_r, lbl_r refer to
    # K_BASELINE (used by Cell 9). Requires Cells 1–7.
    # ------------------------------------------------------------------
    import seaborn as sns
    import matplotlib.patheffects as pe
    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import Normalize
    from scipy.cluster.hierarchy import fcluster
    from scipy.stats import spearmanr
    from sklearn.manifold import TSNE
    from sklearn.metrics import adjusted_rand_score

    # ── SWEEP SETTINGS ───────────────────────────────────────────────────────────
    K_STEP            = 5
    K_SWEEP           = sorted({K_MIN, K_BASELINE, *range(K_STEP, K_MAX + 1, K_STEP)})   # 1, 5, 10, …, 100
    TSNE_FOLLOWS_K    = True       # False → panel (b) fixed at K_TSNE_EFF in every frame
    HEATMAP_VMAX_MODE = "per_k"    # "per_k" (maximum contrast) or "global" (same colour scale in all frames)
    MARK_MODEL_CUTOFF = False      # True → periodic-model traces dotted beyond k_reliable
    SHOW_K            = {K_BASELINE}   # frames displayed inline; the rest are saved and closed
    SAVE_PDF_ALL      = False      # PDF for every frame (PNG is always saved)

    sweep_dir = run_dir / "k_sweep"
    sweep_dir.mkdir(exist_ok=True)

    # ── Highlighted structures ───────────────────────────────────────────────────
    # Chosen from the submitted (faulty) heatmap — re-confirm with k_sweep_summary.csv / Cell 9.
    HIGHLIGHT_REFERENCE   = "mp-2133"
    structures_similar    = ["mp-2133", "cod-2300112", "oqmd-4908"]
    structures_dissimilar = ["cod-1534836", "oqmd-1474198"]
    HIGHLIGHT_COLORS      = {"similar": "blue", "dissimilar": "blue"}   # e.g. "green"/"red" to distinguish
    _missing = [n for n in [HIGHLIGHT_REFERENCE] + structures_similar + structures_dissimilar if n not in _lbl]
    if _missing:
        print(f"  WARNING: highlight targets not in dataset: {_missing}")

    # ── Styling ──────────────────────────────────────────────────────────────────
    FS = {"axis_label": 16, "tick": 16, "legend": 14, "legend_title": 16, "annot": 5,
        "panel_label": 20, "cbar_label": 16, "cif_label": 8}
    LW = {"axis": 2, "tick": 2, "pdd": 0.5, "dendrogram": 1.0, "grid_line": 1.5, "grid_alpha": 0.5}
    HEATMAP_CMAP = "magma_r"
    n_v     = len(valid_names)
    k_axis  = np.arange(1, K_MAX + 1)
    k_ticks = np.linspace(1, K_MAX, num=11, dtype=int)

    def _format_sg(s):
        out = re.sub(r"-(\d)", r"$\\bar{\1}$", s)       # -n → bar notation
        out = re.sub(r"_(\d)", r"$_{\1}$", out)          # screw-axis subscripts
        return out.replace(" ", "")

    sg_lookup  = dict(zip(meta_df["file_name"], meta_df["space_group_symbol"]))
    unique_sgs = sorted({sg_lookup.get(n, "?") for n in valid_names})
    sg_color   = dict(zip(unique_sgs, sns.color_palette("tab20", n_colors=len(unique_sgs))))
    _markers   = ['o', 's', '^', 'D', 'P', 'X', '*', 'v', '<', '>']
    sg_marker  = {sg: _markers[i % len(_markers)] for i, sg in enumerate(unique_sgs)}
    _src       = [get_source(n) for n in valid_names]

    # ── 1. AMD clustering for every k ────────────────────────────────────────────
    def amd_clustering(k):
        X     = np.array([amd_full[n][:k] for n in valid_names], dtype=float)
        cdm   = amd.AMD_pdist(X, metric=AMD_PDIST_METRIC) #AMD_METRIC
        Z     = linkage(cdm, method=LINKAGE_METHOD, optimal_ordering=True)
        order = dendrogram(Z, no_plot=True)["leaves"]
        M     = squareform(cdm)
        return {"cdm": cdm, "Z": Z, "order": order,
                "mat_r": M[np.ix_(order, order)], "lbl_r": [_lbl[i] for i in order],
                "clusters": fcluster(Z, 0.65 * Z[:, 2].max(), criterion="distance")}

    amd_by_k    = {k: amd_clustering(k) for k in K_SWEEP}
    vmax_global = max(v["mat_r"].max() for v in amd_by_k.values())
    for k, v in amd_by_k.items():
        pd.DataFrame(v["mat_r"], index=v["lbl_r"], columns=v["lbl_r"]).to_csv(
            sweep_dir / f"amd_distance_matrix_k{k:03d}.csv")
    print(f"AMD clustering computed for {len(K_SWEEP)} values of k: {K_SWEEP}")

    # ── 2. t-SNE for every distinct k_t (warm-started chain from Cell 7's embedding) ─
    k_tsne_of = {k: (min(k, K_TSNE_EFF) if TSNE_FOLLOWS_K else K_TSNE_EFF) for k in K_SWEEP}
    _ITER_KW  = "max_iter" if "max_iter" in inspect.signature(TSNE.__init__).parameters else "n_iter"

    def _tsne_fit(D, init):
        model = TSNE(n_components=2, metric="precomputed", init=init, perplexity=TSNE_PERPLEXITY_EFF,
                    learning_rate="auto", random_state=42, **{_ITER_KW: 2000})
        return model.fit_transform(D), float(model.kl_divergence_)

    tsne_by_k = {K_TSNE_EFF: (coords, float(tsne.kl_divergence_))}   # Cell 7 embedding, reused exactly
    _prev = coords
    for kt in sorted(set(k_tsne_of.values()) - {K_TSNE_EFF}, reverse=True):
        D_kt  = pairwise_emd(valid_names, pdd_store, kt)
        pd.DataFrame(D_kt, index=_lbl, columns=_lbl).to_csv(sweep_dir / f"pdd_emd_matrix_k{kt:03d}.csv")
        _init = _prev - _prev.mean(axis=0)
        _init = 1e-4 * _init / _init[:, 0].std()          # t-SNE expects a small-scale initialisation
        tsne_by_k[kt] = _tsne_fit(D_kt, _init)
        _prev = tsne_by_k[kt][0]
        pd.DataFrame({"name": _lbl, "tsne_1": _prev[:, 0], "tsne_2": _prev[:, 1]}).to_csv(
            sweep_dir / f"tsne_coords_k{kt:03d}.csv", index=False)
    print(f"t-SNE computed for k_t = {sorted(tsne_by_k)}")

    # ── 3. Plotting ──────────────────────────────────────────────────────────────
    def _highlight(ax, labels, targets, color, lw=2.2, alpha=0.95, text_fs=16):
        idx = {nm: i for i, nm in enumerate(labels)}
        n = len(labels)
        for nm in targets:
            if nm not in idx:
                continue
            x = y = idx[nm] + 0.5
            ax.plot([0, x], [y, y], color=color, lw=lw, alpha=alpha, zorder=5)   # row: left edge → diagonal
            ax.plot([x, x], [n, y], color=color, lw=lw, alpha=alpha, zorder=5)   # column: bottom → diagonal
            ax.text(x, y, nm, color=color, fontsize=text_fs, ha="left", va="bottom", fontweight="bold",
                    zorder=6, path_effects=[pe.withStroke(linewidth=2, foreground="white")])

    def draw_summary(k):
        A, kt = amd_by_k[k], k_tsne_of[k]
        emb   = tsne_by_k[kt][0]
        mat, lbls, Z = A["mat_r"], A["lbl_r"], A["Z"]
        vmax  = vmax_global if HEATMAP_VMAX_MODE == "global" else mat.max()

        fig = plt.figure(figsize=(22, 22))
        outer_gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1, 1.5], hspace=0.2)
        top_gs = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer_gs[0], width_ratios=[1, 1], wspace=0.1)
        bot_gs = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer_gs[1], width_ratios=[1, 2], wspace=0.02)
        ax_pdd, ax_tsne = fig.add_subplot(top_gs[0]), fig.add_subplot(top_gs[1])
        ax_dend, ax_heat = fig.add_subplot(bot_gs[0]), fig.add_subplot(bot_gs[1])

        # (a) PDD rows vs k
        for name in valid_names:
            color = SRC_COLORS.get(get_source(name), "#757575")
            k_cut = min(k_rel[name], K_MAX) if (MARK_MODEL_CUTOFF and model_flags.get(name, False)) else K_MAX
            for row in pdd_store[name]:
                d = row[1:]
                ax_pdd.plot(k_axis[:k_cut], d[:k_cut], color=color, alpha=LW["grid_alpha"],
                            linewidth=LW["pdd"], zorder=1)
                if k_cut < K_MAX:
                    ax_pdd.plot(k_axis[k_cut - 1:], d[k_cut - 1:], color=color, alpha=0.15,
                                linewidth=LW["pdd"], linestyle=":", zorder=0)
        ax_pdd.axvline(k, color="black", linestyle="--", linewidth=2, zorder=3)
        handles_a = [Line2D([0], [0], color=SRC_COLORS[s], linewidth=2, label=fr"{s} ($N={_src.count(s)}$)")
                    for s in ["MP", "COD", "OQMD", "MATTERGEN"] if _src.count(s)]

        # if MARK_MODEL_CUTOFF and model_names:
        #     handles_a.append(Line2D([0], [0], color="#555555", linewidth=1.5, linestyle=":",
        #                             label=r"Periodic model, $k > k_{\mathrm{rel}}$"))
        
        handles_a.append(Line2D([0], [0], color="black", linestyle="--", linewidth=2, label=fr"$k = {k}$"))

        ax_pdd.legend(handles=handles_a, fontsize=FS["legend"], loc="upper left")
        ax_pdd.set_xlabel("Neighbour Index (k)", fontsize=FS["axis_label"])
        ax_pdd.set_ylabel("Neighbour Distance (PDD) [Å]", fontsize=FS["axis_label"])
        ax_pdd.set_xticks(k_ticks)
        ax_pdd.grid(True, linewidth=LW["grid_line"], alpha=LW["grid_alpha"], linestyle="--")
        ax_pdd.tick_params(labelsize=FS["tick"], width=LW["tick"])

        # (b) t-SNE of the PDD EMD matrix at k_t
        for name, (x, y) in zip(valid_names, emb):
            sg = sg_lookup.get(name, "?")
            ax_tsne.scatter(x, y, color=sg_color[sg], marker=sg_marker[sg], s=120, alpha=0.9,
                            edgecolors="black", linewidths=0.8, zorder=3)
        ax_tsne.legend(
            handles=[Line2D([0], [0], marker=sg_marker[sg], color="w", markerfacecolor=sg_color[sg],
                            markeredgecolor="black", markeredgewidth=0.8, markersize=10,
                            label=_format_sg(sg)) for sg in unique_sgs],
            fontsize=FS["legend"] - 2, title="Space Group", title_fontsize=FS["legend_title"] - 2,
            ncol=min(len(unique_sgs), 8), loc="upper center", bbox_to_anchor=(0.5, -0.14),
            frameon=False, columnspacing=1.0, handletextpad=0.3, borderaxespad=0.0)
        ax_tsne.set_zorder(10)             # draw above heatmap/dendrogram axes so the legend is never covered
        ax_tsne.text(0.98, 0.98, fr"PDD EMD, $k = {kt}$", transform=ax_tsne.transAxes,
                    ha="right", va="top", fontsize=FS["legend"])
        ax_tsne.set_xlabel("t-SNE Dim 1", fontsize=FS["axis_label"])
        ax_tsne.set_ylabel("t-SNE Dim 2", fontsize=FS["axis_label"])
        ax_tsne.grid(True, linewidth=LW["grid_line"], alpha=LW["grid_alpha"], linestyle="--")
        ax_tsne.tick_params(labelsize=0, width=LW["tick"])

        # (c) AMD dendrogram (labels in ORIGINAL order; scipy reorders them)
        dendrogram(Z, labels=_lbl, ax=ax_dend, leaf_rotation=0, leaf_font_size=FS["cif_label"],
                color_threshold=0.65 * Z[:, 2].max(), orientation="left")
        for line in ax_dend.get_lines():
            line.set_linewidth(LW["dendrogram"])
        ax_dend.invert_yaxis()

        ax_dend.set_xlabel(f"{LINKAGE_METHOD.capitalize()} Linkage (AMD, {AMD_METRIC.capitalize()}, k={k}) [Å]", fontsize=FS["axis_label"])

        ax_dend.grid(axis="x", linewidth=LW["grid_line"], alpha=LW["grid_alpha"], linestyle="--")
        ax_dend.tick_params(axis="y", labelsize=FS["cif_label"], width=LW["tick"])
        ax_dend.tick_params(axis="x", labelsize=FS["tick"], width=LW["tick"])

        # (d) AMD heatmap (lower triangle)
        sns.heatmap(mat, mask=np.triu(np.ones_like(mat, dtype=bool), k=1), square=True, ax=ax_heat,
                    xticklabels=lbls, yticklabels=lbls, cmap=HEATMAP_CMAP, vmin=0, vmax=vmax,
                    annot=(n_v <= 20), fmt=".3f", annot_kws={"size": FS["annot"]} if n_v <= 20 else {},
                    linewidths=0.1, cbar=False)
        ax_heat.tick_params(axis="x", rotation=90, labelsize=FS["cif_label"], width=LW["tick"])
        ax_heat.tick_params(axis="y", left=True, labelleft=False, width=LW["tick"])
        sm = ScalarMappable(cmap=HEATMAP_CMAP, norm=Normalize(vmin=0, vmax=vmax))
        sm.set_array([])
        cb = fig.colorbar(sm, cax=ax_heat.inset_axes([0.42, 0.84, 0.54, 0.04]), orientation="horizontal")
        
        cb.set_label(f"AMD Distance, {AMD_METRIC.capitalize()} [Å] (k={k})", fontsize=FS["cbar_label"])

        cb.ax.tick_params(labelsize=FS["tick"], width=LW["tick"])

        _highlight(ax_heat, lbls, structures_similar,    HIGHLIGHT_COLORS["similar"])
        _highlight(ax_heat, lbls, structures_dissimilar, HIGHLIGHT_COLORS["dissimilar"])

        for ax in [ax_pdd, ax_tsne, ax_dend, ax_heat]:
            for spine in ax.spines.values():
                spine.set_linewidth(LW["axis"])

        # Align dendrogram with heatmap
        fig.canvas.draw()
        ph, pdn = ax_heat.get_position(), ax_dend.get_position()
        ax_dend.set_position([pdn.x0, ph.y0, pdn.width, ph.height])

        for ax, lab, x_off in [(ax_pdd, "(a)", -0.10), (ax_tsne, "(b)", -0.10),
                            (ax_dend, "(c)", -0.10), (ax_heat, "(d)", 0.0)]:
            ax.text(x_off, 1.04, lab, transform=ax.transAxes, fontsize=FS["panel_label"],
                    fontweight="bold", va="top", ha="left")
        return fig

    # ── 4. Sweep: draw, save, summarise ──────────────────────────────────────────
    base     = amd_by_k[K_BASELINE]
    ref_idx  = _lbl.index(HIGHLIGHT_REFERENCE) if HIGHLIGHT_REFERENCE in _lbl else None
    _targets = [n for n in dict.fromkeys(structures_similar + structures_dissimilar)
                if n in _lbl and n != HIGHLIGHT_REFERENCE]
    sweep_rows = []

    for k in K_SWEEP:
        fig   = draw_summary(k)
        fname = f"summary_4panel_k{k:03d}"
        fig.savefig(sweep_dir / f"{fname}.png", dpi=300, bbox_inches="tight")
        fig.savefig(sweep_dir / f"{fname}.pdf", bbox_inches="tight")
        if SAVE_PDF_ALL:
            fig.savefig(sweep_dir / f"{fname}.pdf", bbox_inches="tight")
        if k == K_BASELINE:
            fig.savefig(run_dir / f"{fname}.png", dpi=300, bbox_inches="tight")
            fig.savefig(run_dir / f"{fname}.pdf", bbox_inches="tight")
        if k in SHOW_K:
            plt.show()
        plt.close(fig)

        A = amd_by_k[k]
        row = {"k": k, "k_tsne": k_tsne_of[k], "tsne_KL": tsne_by_k[k_tsne_of[k]][1],
            "n_clusters_0.65max": int(len(set(A["clusters"]))),
            "ARI_vs_baseline": float(adjusted_rand_score(base["clusters"], A["clusters"])),
            "spearman_vs_baseline": float(spearmanr(base["cdm"], A["cdm"]).correlation),
            "max_AMD_distance_A": float(A["cdm"].max())}
        if ref_idx is not None:                       # rank of highlighted structures relative to the reference
            d_ref  = squareform(A["cdm"])[ref_idx]
            ranked = sorted((j for j in range(n_v) if j != ref_idx), key=lambda j: d_ref[j])
            rank   = {_lbl[j]: r + 1 for r, j in enumerate(ranked)}
            row.update({f"rank_{t}": rank[t] for t in _targets})
        sweep_rows.append(row)
        print(f"  k = {k:3d}  (t-SNE k = {k_tsne_of[k]:3d})  saved {fname}.png")

    sweep_df = pd.DataFrame(sweep_rows)
    sweep_df.to_csv(run_dir / "k_sweep_summary.csv", index=False)
    with pd.option_context("display.width", 220, "display.float_format", "{:.3f}".format):
        print(f"\\nStability across k (baseline k = {K_BASELINE}; ranks relative to {HIGHLIGHT_REFERENCE}, "
            f"out of {n_v - 1}):")
        print(sweep_df.to_string(index=False))

    # ── 5. Baseline objects for Cell 9 ───────────────────────────────────────────
    cdm_amd, Z_amd, order = base["cdm"], base["Z"], base["order"]
    mat_amd, mat_r, lbl_r = squareform(base["cdm"]), base["mat_r"], base["lbl_r"]
    K_TSNE_MAIN_FIG       = k_tsne_of[K_BASELINE]
    print(f"\\nMain figure: k = {K_BASELINE}; panel (b) t-SNE at k = {K_TSNE_MAIN_FIG}  → state in caption")

else:
    # %% [Cell 8] Summary figure (four panels)
    #  (a) PDD rows vs k (neighbour distance, Å). Periodic models: solid to k_reliable, dotted beyond.
    #  (b) t-SNE of the PDD EMD matrix (k = K_TSNE_EFF), by space group
    #  (c) AMD dendrogram (AMD_pdist, L∞, k = K_BASELINE)
    #  (d) AMD distance heatmap, lower triangle, in dendrogram leaf order
    import seaborn as sns
    import matplotlib.patheffects as pe
    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import Normalize

    def _format_sg(s):
        out = re.sub(r"-(\\d)", r"$\\\\bar{\\1}$", s)       # -n → bar notation
        out = re.sub(r"_(\\d)", r"$_{\\1}$", out)          # screw-axis subscripts
        return out.replace(" ", "")

    FS = {"axis_label": 16, "tick": 16, "legend": 14, "legend_title": 16, "annot": 5,
        "panel_label": 20, "cbar_label": 16, "cif_label": 10}
    LW = {"axis": 2, "tick": 2, "pdd": 0.5, "dendrogram": 1.0, "grid_line": 1.5, "grid_alpha": 0.5}
    HEATMAP_CMAP = "magma_r"
    n_v = len(valid_names)

    MARK_MODEL_CUTOFF = False      # True → periodic-model traces dotted beyond k_reliable

    # ── AMD distance matrix + shared dendrogram ordering ─────────────────────────
    amd_vecs  = np.array([amd_desc_by_k[K_BASELINE][n] for n in valid_names], dtype=float)
    cdm_amd   = amd.AMD_pdist(amd_vecs, metric=AMD_PDIST_METRIC) # AMD_METRIC
    Z_amd     = linkage(cdm_amd, method=LINKAGE_METHOD, optimal_ordering=True)
    mat_amd   = squareform(cdm_amd)
    order     = dendrogram(Z_amd, no_plot=True)["leaves"]
    mat_r     = mat_amd[np.ix_(order, order)]
    lbl_r     = [_lbl[i] for i in order]
    pd.DataFrame(mat_r, index=lbl_r, columns=lbl_r).to_csv(run_dir / f"amd_distance_matrix_k{K_BASELINE:03d}.csv")

    k_axis  = np.arange(1, K_MAX + 1)
    k_ticks = np.linspace(1, K_MAX, num=11, dtype=int)

    # ── Space-group colours/markers (one mapping for scatter AND legend) ─────────
    sg_lookup  = dict(zip(meta_df["file_name"], meta_df["space_group_symbol"]))
    unique_sgs = sorted({sg_lookup.get(n, "?") for n in valid_names})
    sg_color   = dict(zip(unique_sgs, sns.color_palette("tab20", n_colors=len(unique_sgs))))
    _markers   = ['o', 's', '^', 'D', 'P', 'X', '*', 'v', '<', '>']
    sg_marker  = {sg: _markers[i % len(_markers)] for i, sg in enumerate(unique_sgs)}

    # ── Layout ───────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(22, 22))
    outer_gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1, 1.5], hspace=0.2)
    top_gs = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer_gs[0], width_ratios=[1, 1], wspace=0.1)
    bot_gs = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer_gs[1], width_ratios=[1, 2], wspace=0.02)
    ax_pdd, ax_tsne = fig.add_subplot(top_gs[0]), fig.add_subplot(top_gs[1])
    ax_dend, ax_heat = fig.add_subplot(bot_gs[0]), fig.add_subplot(bot_gs[1])

    # ── (a) PDD rows vs k ────────────────────────────────────────────────────────
    for name in valid_names:
        color = SRC_COLORS.get(get_source(name), "#757575")
        k_cut = min(k_rel[name], K_MAX) if model_flags.get(name, False) else K_MAX
        for row in pdd_store[name]:
            d = row[1:]
            ax_pdd.plot(k_axis[:k_cut], d[:k_cut], color=color, alpha=LW["grid_alpha"],
                        linewidth=LW["pdd"], zorder=1)
            if k_cut < K_MAX:
                ax_pdd.plot(k_axis[k_cut - 1:], d[k_cut - 1:], color=color, alpha=0.15,
                            linewidth=LW["pdd"], linestyle=":", zorder=0)

    _src = [get_source(n) for n in valid_names]
    handles_a = [Line2D([0], [0], color=SRC_COLORS[s], linewidth=2, label=fr"{s} ($N={_src.count(s)}$)")
                for s in ["MP", "COD", "OQMD", "MATTERGEN"] if _src.count(s)]
    
    # if MARK_MODEL_CUTOFF and model_names:
    #     handles_a.append(Line2D([0], [0], color="#555555", linewidth=1.5, linestyle=":",
    #                             label=r"Periodic model, $k > k_{\mathrm{rel}}$"))
        
    ax_pdd.legend(handles=handles_a, fontsize=FS["legend"], loc="upper left")
    ax_pdd.set_xlabel("Neighbour Index (k)", fontsize=FS["axis_label"])
    ax_pdd.set_ylabel("Neighbour Distance (PDD) [Å]", fontsize=FS["axis_label"])
    ax_pdd.set_xticks(k_ticks)
    ax_pdd.grid(True, linewidth=LW["grid_line"], alpha=LW["grid_alpha"], linestyle="--")
    ax_pdd.tick_params(labelsize=FS["tick"], width=LW["tick"])

    # ── (b) t-SNE ────────────────────────────────────────────────────────────────
    for name, (x, y) in zip(valid_names, coords):
        sg = sg_lookup.get(name, "?")
        ax_tsne.scatter(x, y, color=sg_color[sg], marker=sg_marker[sg], s=120, alpha=0.9,
                        edgecolors="black", linewidths=0.8, zorder=3)
    ax_tsne.legend(
        handles=[Line2D([0], [0], marker=sg_marker[sg], color="w", markerfacecolor=sg_color[sg],
                        markeredgecolor="black", markeredgewidth=0.8, markersize=10,
                        label=_format_sg(sg)) for sg in unique_sgs],
        fontsize=FS["legend"] - 2, title="Space Group", title_fontsize=FS["legend_title"] - 2,
        ncol=min(len(unique_sgs), 8), loc="upper center", bbox_to_anchor=(0.5, -0.14),
        frameon=False, columnspacing=1.0, handletextpad=0.3, borderaxespad=0.0)
    ax_tsne.set_xlabel("t-SNE Dim 1", fontsize=FS["axis_label"])
    ax_tsne.set_ylabel("t-SNE Dim 2", fontsize=FS["axis_label"])
    ax_tsne.grid(True, linewidth=LW["grid_line"], alpha=LW["grid_alpha"], linestyle="--")
    ax_tsne.tick_params(labelsize=0, width=LW["tick"])

    # ── (c) AMD dendrogram (labels in ORIGINAL order; scipy reorders them) ───────
    dendrogram(Z_amd, labels=_lbl, ax=ax_dend, leaf_rotation=0, leaf_font_size=FS["cif_label"],
            color_threshold=0.65 * max(Z_amd[:, 2]), orientation="left")
    for line in ax_dend.get_lines():
        line.set_linewidth(LW["dendrogram"])
    ax_dend.invert_yaxis()

    ax_dend.set_xlabel(f"{LINKAGE_METHOD.capitalize()} Linkage (AMD, {AMD_METRIC.capitalize()}, k={K_BASELINE}) [Å]", fontsize=FS["axis_label"])

    ax_dend.grid(axis="x", linewidth=LW["grid_line"], alpha=LW["grid_alpha"], linestyle="--")
    ax_dend.tick_params(axis="y", labelsize=FS["cif_label"], width=LW["tick"])
    ax_dend.tick_params(axis="x", labelsize=FS["tick"], width=LW["tick"])

    # ── (d) AMD heatmap (lower triangle) ─────────────────────────────────────────
    sns.heatmap(mat_r, mask=np.triu(np.ones_like(mat_r, dtype=bool), k=1), square=True, ax=ax_heat,
                xticklabels=lbl_r, yticklabels=lbl_r, cmap=HEATMAP_CMAP, vmin=0, vmax=mat_r.max(),
                annot=(n_v <= 20), fmt=".3f", annot_kws={"size": FS["annot"]} if n_v <= 20 else {},
                linewidths=0.1, cbar=False)
    ax_heat.tick_params(axis="x", rotation=90, labelsize=FS["cif_label"], width=LW["tick"])
    ax_heat.tick_params(axis="y", left=True, labelleft=False, width=LW["tick"])

    sm = ScalarMappable(cmap=HEATMAP_CMAP, norm=Normalize(vmin=0, vmax=mat_r.max()))
    sm.set_array([])
    cax = ax_heat.inset_axes([0.42, 0.84, 0.54, 0.04])
    cb  = fig.colorbar(sm, cax=cax, orientation="horizontal")

    cb.set_label(f"AMD Distance, {AMD_METRIC.capitalize()} [Å] (k={K_BASELINE})", fontsize=FS["cbar_label"])

    cb.ax.tick_params(labelsize=FS["tick"], width=LW["tick"])

    for ax in [ax_pdd, ax_tsne, ax_dend, ax_heat]:
        for spine in ax.spines.values():
            spine.set_linewidth(LW["axis"])

    # ── Highlighted structures ───────────────────────────────────────────────────
    # Chosen from the submitted (faulty) heatmap — re-confirm with Cell 9 before finalising.
    structures_similar    = ["mp-2133", "cod-2300112", "oqmd-4908"]
    structures_dissimilar = ["cod-1534836", "oqmd-1474198"]
    HIGHLIGHT_COLORS      = {"similar": "blue", "dissimilar": "blue"}   # e.g. "green"/"red" to distinguish

    def _highlight(ax, labels, targets, color, lw=2.2, alpha=0.95, text_fs=16):
        idx = {nm: i for i, nm in enumerate(labels)}
        n = len(labels)
        for nm in targets:
            if nm not in idx:
                print(f"  WARNING: highlight target '{nm}' not in heatmap labels")
                continue
            x = y = idx[nm] + 0.5
            ax.plot([0, x], [y, y], color=color, lw=lw, alpha=alpha, zorder=5)   # row: left edge → diagonal
            ax.plot([x, x], [n, y], color=color, lw=lw, alpha=alpha, zorder=5)   # column: bottom → diagonal
            ax.text(x, y, nm, color=color, fontsize=text_fs, ha="left", va="bottom", fontweight="bold",
                    zorder=6, path_effects=[pe.withStroke(linewidth=2, foreground="white")])

    _highlight(ax_heat, lbl_r, structures_similar,    HIGHLIGHT_COLORS["similar"])
    _highlight(ax_heat, lbl_r, structures_dissimilar, HIGHLIGHT_COLORS["dissimilar"])

    # ── Align dendrogram with heatmap ────────────────────────────────────────────
    fig.canvas.draw()
    ph, pdn = ax_heat.get_position(), ax_dend.get_position()
    ax_dend.set_position([pdn.x0, ph.y0, pdn.width, ph.height])

    for ax, lab, x_off in [(ax_pdd, "(a)", -0.10), (ax_tsne, "(b)", -0.10),
                        (ax_dend, "(c)", -0.10), (ax_heat, "(d)", 0.0)]:
        ax.text(x_off, 1.04, lab, transform=ax.transAxes, fontsize=FS["panel_label"],
                fontweight="bold", va="top", ha="left")

    fig.savefig(run_dir / f"summary_4panel_k{K_BASELINE:03d}.png", dpi=300, bbox_inches="tight")
    fig.savefig(run_dir / f"summary_4panel_k{K_BASELINE:03d}.pdf", bbox_inches="tight")
    plt.show()
    print(f"Saved summary_4panel_k{K_BASELINE:03d}  (caption: t-SNE on PDD EMD at k = {K_TSNE_EFF})")




In [ ]:
# %% [Cell 9] Distances for highlighted structures + re-verification of selections
REFERENCE = "mp-2133"
selection = list(dict.fromkeys([REFERENCE] + structures_similar + structures_dissimilar))

amd_df  = pd.DataFrame(mat_r, index=lbl_r, columns=lbl_r)
emd_df  = pd.DataFrame(emd_tsne, index=_lbl, columns=_lbl)
present = [n for n in selection if n in amd_df.index]
if set(selection) - set(present):
    print("Missing from dataset:", sorted(set(selection) - set(present)))

amd_details = amd_df.loc[present, present]
pdd_details = emd_df.loc[present, present]
print(f"AMD distance ({AMD_METRIC}, k={K_BASELINE}) [Å]:\n{amd_details.round(4)}\n")
print(f"PDD EMD ({AMD_PDIST_METRIC}, k={K_TSNE_EFF}) [Å]:\n{pdd_details.round(4)}\n")

if REFERENCE in amd_df.index:
    ranking = amd_df[REFERENCE].drop(REFERENCE).sort_values()
    print(f"Nearest to {REFERENCE}:\n{ranking.head(5).round(4)}\n")
    print(f"Farthest from {REFERENCE}:\n{ranking.tail(5).round(4)}\n")
    for nm in structures_similar + structures_dissimilar:
        if nm in ranking.index:
            print(f"  {nm:16s} rank {ranking.index.get_loc(nm) + 1:3d} / {len(ranking)}   d = {ranking[nm]:.4f} Å")

amd_details.to_csv(run_dir / "highlighted_amd_distances.csv")
pdd_details.to_csv(run_dir / "highlighted_pdd_emd.csv")


In [ ]:
# %% [Cell 9b] Deduplication: groups, representatives, validation, sensitivity, species test
# ------------------------------------------------------------------
# Rule: DEDUP_LINKAGE linkage on the AMD distance (first DEDUP_K values, metric DEDUP_METRIC),
#       cut at DEDUP_TAU_A. With "complete" linkage every pair in a group is within DEDUP_TAU_A.
# Optional confirmation: PDD EMD at k = CONFIRM_K with the SAME ground metric, same threshold.
#       ||AMD_a − AMD_b|| ≤ EMD_k ≤ EMD_K (k ≤ K) for any norm, so the AMD step is lossless.
# Representative: group medoid (smallest summed distance to the other members);
#       ties → SOURCE_PRIORITY, then identifier.
# Reference: pymatgen StructureMatcher (species-aware) at two strictness levels.
# Requires Cells 1–9.
# ------------------------------------------------------------------
import warnings
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import adjusted_rand_score
from pymatgen.analysis.structure_matcher import StructureMatcher

# ── SETTINGS ─────────────────────────────────────────────────────────────────
DEDUP_METRIC    = "euclidean"          # "euclidean" (as in the manuscript) or "chebyshev" (L∞)
DEDUP_K         = 15
DEDUP_TAU_A     = 0.1
DEDUP_LINKAGE   = "complete"           # "complete": every pair ≤ τ | "single": connected components
                                       # "ward": dendrogram cut (Euclidean only; NOT a pairwise bound)
CONFIRM_K       = min(K_MAX, max(17, K_TSNE_EFF))  # ≥ 17: first neighbour differing between ideal wurtzite and zinc blende
SHOW_WARD_CUT   = True                 # also report the Fig. 3(c) Ward dendrogram cut at τ (Euclidean only)
SENS_METRICS    = ("euclidean", "chebyshev")   # metrics included in the sensitivity sweep
SOURCE_PRIORITY = {"COD": 0, "MP": 1, "OQMD": 2, "MATTERGEN": 3, "other": 4}
TAU_GRID        = [0.0, 1e-3, 3e-3, 0.01, 0.02, 0.03, 0.05, 0.1, 0.2, 0.3]

# k values in the sweep: always include DEDUP_K and CONFIRM_K (needed by the plots)
K_GRID    = sorted({k for k in [5, 10, 15, 17, 20, 25, 30, 50, 100, DEDUP_K, CONFIRM_K] if 1 <= k <= K_MAX})
K_PANEL_A = sorted({k for k in [5, 15, 30, 100, DEDUP_K] if k in K_GRID})   # AMD curves in panel (a)

METRIC_NAME = {"euclidean": "Euclidean (L2)", "chebyshev": "Chebyshev (L∞)"}
if DEDUP_METRIC not in METRIC_NAME:
    raise ValueError(f"DEDUP_METRIC must be one of {list(METRIC_NAME)}")
if DEDUP_LINKAGE == "ward" and DEDUP_METRIC != "euclidean":
    raise ValueError("Ward linkage requires DEDUP_METRIC = 'euclidean'")
if not 1 <= DEDUP_K <= K_MAX:
    raise ValueError(f"DEDUP_K must be between 1 and K_MAX = {K_MAX}")

REFERENCES = {
    "type": StructureMatcher(),                                                  # pymatgen defaults, scale=True
    "geom": StructureMatcher(ltol=0.01, stol=0.02, angle_tol=0.5, scale=False),  # geometric, ~τ-scale
}

dedup_dir = run_dir / "dedup"
dedup_dir.mkdir(exist_ok=True)
n_s = len(valid_names)
iu  = np.triu_indices(n_s, 1)
sg  = dict(zip(meta_df["file_name"], meta_df["space_group_symbol"]))
vpa = dict(zip(meta_df["file_name"], meta_df["volume_per_atom_A3"]))
tag_m = "L2" if DEDUP_METRIC == "euclidean" else "Linf"

# ── HELPERS ──────────────────────────────────────────────────────────────────
def amd_dist(k, metric=DEDUP_METRIC):
    """Square matrix of AMD distances using the first k AMD values."""
    X = np.array([amd_full[n][:k] for n in valid_names])
    return squareform(amd.AMD_pdist(X, metric=metric))

def emd_m(D1, D2, k, metric=DEDUP_METRIC):
    """PDD EMD truncated to k neighbours, with the given ground metric."""
    return float(amd.EMD(np.ascontiguousarray(D1[:, :k + 1]),
                         np.ascontiguousarray(D2[:, :k + 1]), metric=metric))

_pdd_cache = {}
def pdd_dist(k, metric=DEDUP_METRIC):
    """Square matrix of PDD EMDs at k (cached)."""
    if (k, metric) not in _pdd_cache:
        M = np.zeros((n_s, n_s))
        for i, j in zip(*iu):
            M[i, j] = M[j, i] = emd_m(pdd_store[valid_names[i]], pdd_store[valid_names[j]], k, metric)
        _pdd_cache[(k, metric)] = M
    return _pdd_cache[(k, metric)]

def prefix_dist(diff, metric=DEDUP_METRIC):
    """AMD distance using the first k components, for every k = 1…K."""
    return np.sqrt(np.cumsum(diff ** 2)) if metric == "euclidean" else np.maximum.accumulate(diff)

def group_labels(Dsq, tau, method=DEDUP_LINKAGE):
    return fcluster(linkage(squareform(Dsq, checks=False), method), tau, criterion="distance")

def coord_number(D, f=1.15):
    """Weighted mean number of neighbours within f × nearest-neighbour distance (from the PDD)."""
    d = D[:, 1:]
    return float(D[:, 0] @ (d <= f * d[:, [0]]).sum(axis=1))

cn = {n: coord_number(pdd_store[n]) for n in valid_names}

# ── 0. Precise interpretation of the threshold ───────────────────────────────
_sqrt_k = np.sqrt(DEDUP_K) if DEDUP_METRIC == "euclidean" else 1.0
threshold_interpretation = {
    "metric": METRIC_NAME[DEDUP_METRIC], "k": DEDUP_K, "tau_A": DEDUP_TAU_A, "linkage": DEDUP_LINKAGE,
    "max_difference_any_single_AMD_component_A": DEDUP_TAU_A,
    "max_rms_difference_over_k_components_A": DEDUP_TAU_A / np.sqrt(DEDUP_K) if DEDUP_METRIC == "euclidean" else DEDUP_TAU_A,
    "always_merged_if_every_atom_displaced_by_at_most_A": DEDUP_TAU_A / (2 * _sqrt_k),
    "pairwise_guarantee": {"complete": "every pair in a group is within tau",
                           "single": "groups are connected by chains of pairs within tau",
                           "ward": "none (variance-based merge cost)"}[DEDUP_LINKAGE],
}
print("Threshold interpretation:")
for key, v in threshold_interpretation.items():
    print(f"  {key}: {v:.4g}" if isinstance(v, float) else f"  {key}: {v}")

# ── 1. Independent reference: StructureMatcher (pairwise) ────────────────────
print("\nStructureMatcher references (species-aware, pairwise)…")
ref_match, ref_labels = {}, {}
for rname, matcher in REFERENCES.items():
    M = np.eye(n_s, dtype=bool)
    for i, j in zip(*iu):
        try:
            M[i, j] = M[j, i] = matcher.fit(all_structures[valid_names[i]], all_structures[valid_names[j]])
        except Exception as e:
            print(f"  {rname}: {_lbl[i]} vs {_lbl[j]} failed: {e}")
    ref_match[rname] = M
    n_comp, ref_labels[rname] = connected_components(M, directed=False)
    print(f"  {rname}: {int(M[iu].sum())} matching pairs → {n_comp} groups")

def evaluate(labels):
    same = labels[:, None] == labels[None, :]
    out = {"n_groups": int(len(set(labels)))}
    for rname, M in ref_match.items():
        tp = int((same & M)[iu].sum()); fp = int((same & ~M)[iu].sum()); fn = int((~same & M)[iu].sum())
        out.update({f"{rname}_false_merges": fp, f"{rname}_missed": fn,
                    f"{rname}_precision": tp / (tp + fp) if tp + fp else 1.0,
                    f"{rname}_recall":    tp / (tp + fn) if tp + fn else 1.0,
                    f"{rname}_ARI": float(adjusted_rand_score(ref_labels[rname], labels))})
    return out

# ── 2. Headline rules ────────────────────────────────────────────────────────
D_amd = amd_dist(DEDUP_K)
D_pdd = pdd_dist(CONFIRM_K)
viol  = float((D_amd - D_pdd)[iu].max())
tol_b = 2 * COLLAPSE_TOL * (np.sqrt(CONFIRM_K) if DEDUP_METRIC == "euclidean" else 1.0)
print(f"\nLower bound AMD(k={DEDUP_K}) ≤ PDD EMD(k={CONFIRM_K}), {METRIC_NAME[DEDUP_METRIC]}: "
      f"max violation {viol:.1e} Å ({'holds' if viol <= tol_b else 'CHECK'}; row-merging tolerance {tol_b:.1e} Å)")

RULES = {
    "AMD":     (D_amd, f"AMD {tag_m}, k={DEDUP_K}", DEDUP_LINKAGE),
    "AMD+PDD": (np.maximum(D_amd, D_pdd),
                f"AMD {tag_m} k={DEDUP_K} pre-filter → PDD EMD {tag_m} k={CONFIRM_K}", DEDUP_LINKAGE),
}
if SHOW_WARD_CUT and DEDUP_METRIC == "euclidean" and DEDUP_LINKAGE != "ward":
    RULES["Ward cut"] = (D_amd, f"Ward dendrogram (Fig. 3c), AMD L2 k={DEDUP_K}, cut at τ", "ward")

def group_tables(labels, Dsq):
    members, groups = [], []
    for g in sorted(set(labels)):
        idx    = np.where(labels == g)[0]
        others = np.where(labels != g)[0]
        sums   = Dsq[np.ix_(idx, idx)].sum(axis=1)
        order  = sorted(range(len(idx)), key=lambda t: (round(sums[t], 12),
                        SOURCE_PRIORITY[get_source(valid_names[idx[t]])], _lbl[idx[t]]))
        rep    = idx[order[0]]
        sub    = np.ix_(idx, idx)
        groups.append({
            "group": int(g), "representative": _lbl[rep], "n_members": len(idx),
            "members": "; ".join(_lbl[i] for i in idx),
            "space_groups": "; ".join(sorted({sg[valid_names[i]] for i in idx})),
            "max_intra_A": float(Dsq[sub].max()),
            "min_to_other_groups_A": float(Dsq[np.ix_(idx, others)].min()) if len(others) else np.nan,
            "all_pairs_match_type": bool(ref_match["type"][sub].all()),
            "all_pairs_match_geom": bool(ref_match["geom"][sub].all()),
        })
        for i in idx:
            nm = valid_names[i]
            members.append({"group": int(g), "structure": _lbl[i], "is_representative": i == rep,
                            "source": get_source(nm), "space_group": sg[nm],
                            "n_sites": all_structures[nm].num_sites, "coordination": round(cn[nm], 2),
                            "volume_per_atom_A3": vpa[nm], "distance_to_rep_A": float(Dsq[i, rep])})
    return pd.DataFrame(groups), pd.DataFrame(members)

def pair_audit(labels, Dsq):
    same, rows = labels[:, None] == labels[None, :], []
    for i, j in zip(*iu):
        a, b = valid_names[i], valid_names[j]
        for rname, M in ref_match.items():
            if same[i, j] == M[i, j]:
                continue
            pref  = prefix_dist(np.abs(amd_full[a] - amd_full[b]))
            k_sep = int(np.argmax(pref > DEDUP_TAU_A)) + 1 if (pref > DEDUP_TAU_A).any() else None
            if same[i, j]:
                kind = "false merge"
                if sg[a] == sg[b]:                    cat = "same space group"
                elif abs(cn[a] - cn[b]) < 0.5:        cat = "same coordination, different space group (stacking/ordering variant)"
                else:                                 cat = "different coordination (distinct polymorph)"
            else:
                kind = "missed duplicate"
                dv   = abs(vpa[a] - vpa[b]) / min(vpa[a], vpa[b])
                cat  = f"volume/atom differs by {100 * dv:.1f}%" + (" (scale, e.g. DFT vs experiment)" if dv > 0.01 else "")
            rows.append({"reference": rname, "type": kind, "a": _lbl[i], "b": _lbl[j],
                         "sg_a": sg[a], "sg_b": sg[b], "distance_A": float(Dsq[i, j]),
                         "AMD_first_k_exceeding_tau": k_sep, "category": cat})
    return pd.DataFrame(rows)

dedup_results = {}
for key, (Dsq, desc, method) in RULES.items():
    lab = group_labels(Dsq, DEDUP_TAU_A, method)
    gdf, mdf = group_tables(lab, Dsq)
    adf = pair_audit(lab, Dsq)
    ev  = evaluate(lab)
    split_close = int(((Dsq <= DEDUP_TAU_A) & (lab[:, None] != lab[None, :]))[iu].sum())
    tag = f"{key.replace(' ', '_').replace('+', '_')}_{tag_m}_{method}_k{DEDUP_K:03d}_tau{DEDUP_TAU_A:g}"
    gdf.to_csv(dedup_dir / f"groups_{tag}.csv", index=False)
    mdf.to_csv(dedup_dir / f"members_{tag}.csv", index=False)
    adf.to_csv(dedup_dir / f"pair_audit_{tag}.csv", index=False)
    dedup_results[key] = {"rule": desc, "linkage": method, **ev,
                          "pairs_within_tau_split": split_close,
                          "max_intra_group_distance_A": float(gdf["max_intra_A"].max()),
                          "representatives": gdf["representative"].tolist()}
    print(f"\n[{desc}, τ = {DEDUP_TAU_A} Å, {method} linkage]  {ev['n_groups']} groups from {n_s}")
    print(f"  representatives: {gdf['representative'].tolist()}")
    print(f"  largest within-group distance: {gdf['max_intra_A'].max():.4f} Å"
          + ("  ← exceeds τ (no pairwise guarantee)" if gdf["max_intra_A"].max() > DEDUP_TAU_A + 1e-12 else ""))
    for r in REFERENCES:
        print(f"  vs {r:4s}: false merges {ev[f'{r}_false_merges']:3d}, missed {ev[f'{r}_missed']:4d}, "
              f"precision {ev[f'{r}_precision']:.3f}, recall {ev[f'{r}_recall']:.3f}, ARI {ev[f'{r}_ARI']:.3f}")
    print(f"  pairs within τ split across groups: {split_close}")
    fm = adf[adf["type"] == "false merge"] if len(adf) else adf
    if len(fm):
        print("  False merges:")
        print(fm.drop_duplicates(["a", "b"]).to_string(index=False))

# ── 3. Polymorph check: wurtzite vs zinc blende ──────────────────────────────
for a, b in [("mp-2133.cif", "mp-1986.cif")]:
    if a in amd_full and b in amd_full:
        pref = prefix_dist(np.abs(amd_full[a] - amd_full[b]))
        print(f"\n{short_label(a)} ({sg[a]}) vs {short_label(b)} ({sg[b]}), {METRIC_NAME[DEDUP_METRIC]}: "
              f"AMD k={DEDUP_K} {pref[DEDUP_K - 1]:.4f} | k=15 {pref[14]:.4f} | k=16 {pref[15]:.4f} | "
              f"k=17 {pref[16]:.4f} Å; PDD EMD k=17 {emd_m(pdd_store[a], pdd_store[b], 17):.4f} Å; "
              f"StructureMatcher(type) = {ref_match['type'][valid_names.index(a), valid_names.index(b)]}")

# ── 4. Threshold × k × metric sensitivity ────────────────────────────────────
metrics_sens = [m for m in SENS_METRICS if not (DEDUP_LINKAGE == "ward" and m != "euclidean")]
print(f"\nSensitivity sweep: metrics {metrics_sens}, k = {K_GRID}, τ = {TAU_GRID}")
sens_rows = []
for metric in metrics_sens:
    for desc in ["AMD", "PDD"]:
        for k in K_GRID:
            Dk = amd_dist(k, metric) if desc == "AMD" else pdd_dist(k, metric)
            Zk = linkage(squareform(Dk, checks=False), DEDUP_LINKAGE)
            for tau in TAU_GRID:
                sens_rows.append({"metric": metric, "descriptor": desc, "k": k, "tau_A": tau,
                                  **evaluate(fcluster(Zk, tau, criterion="distance"))})
sens_df = pd.DataFrame(sens_rows)
sens_df.to_csv(dedup_dir / "threshold_sensitivity.csv", index=False)

def _select(metric, desc, k):
    """Rows of sens_df for one curve; warns instead of silently plotting nothing."""
    s = sens_df[(sens_df.metric == metric) & (sens_df.descriptor == desc) & (sens_df.k == k)]
    if s.empty:
        warnings.warn(f"No sensitivity data for {desc}, k={k}, {metric} — curve omitted")
    return s.sort_values("tau_A")

fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) number of groups
for k in K_PANEL_A:
    s = _select(DEDUP_METRIC, "AMD", k)
    axs[0].plot(s.tau_A, s.n_groups, marker="o", ms=3, lw=2 if k == DEDUP_K else 1,
                label=f"AMD, k={k}" + (" (rule)" if k == DEDUP_K else ""))
s = _select(DEDUP_METRIC, "PDD", CONFIRM_K)
axs[0].plot(s.tau_A, s.n_groups, "k--", marker="s", ms=4, lw=1.8, zorder=4, label=f"PDD EMD, k={CONFIRM_K}")
for m_alt in [m for m in metrics_sens if m != DEDUP_METRIC]:
    s = _select(m_alt, "AMD", DEDUP_K)
    axs[0].plot(s.tau_A, s.n_groups, color="grey", ls=":", marker="^", ms=3,
                label=f"AMD, k={DEDUP_K}, {METRIC_NAME[m_alt]}")

# (b) false merges and (c) missed duplicates — PDD drawn on top so overlapping curves stay visible
CURVES = [("AMD", DEDUP_K,   "#1565C0", "o", 1.2, 2),
          ("PDD", CONFIRM_K, "black",   "s", 1.8, 4)]
for desc, k, col, mk, lw, zo in CURVES:
    s = _select(DEDUP_METRIC, desc, k)
    for r, ls in [("geom", "-"), ("type", ":")]:
        kw = dict(color=col, ls=ls, marker=mk, ms=4, lw=lw, zorder=zo, mfc="none" if r == "type" else col)
        axs[1].plot(s.tau_A, s[f"{r}_false_merges"], label=f"{desc} k={k}, {r}", **kw)
        axs[2].plot(s.tau_A, s[f"{r}_missed"],       label=f"{desc} k={k}, {r}", **kw)

for ax, yl in zip(axs, ["Number of groups", "False merges (pairs)", "Missed duplicates (pairs)"]):
    ax.set_xscale("symlog", linthresh=1e-3)
    ax.axvline(DEDUP_TAU_A, color="grey", ls="--", lw=1)
    ax.set_xlabel(f"Threshold τ [Å] ({METRIC_NAME[DEDUP_METRIC]})")
    ax.set_ylabel(yl)
    ax.grid(True, ls="--", alpha=0.5)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(dedup_dir / f"figS_dedup_sensitivity_{tag_m}.png", dpi=300, bbox_inches="tight")
fig.savefig(dedup_dir / f"figS_dedup_sensitivity_{tag_m}.pdf", bbox_inches="tight")
plt.show()

# ── 5. Species handling: same positions, different chemistry ─────────────────
species_test = {}
if "mp-2229.cif" in all_structures:
    rs = SpacegroupAnalyzer(all_structures["mp-2229.cif"]).get_conventional_standard_structure()
    # Rock-salt sites lie on a simple-cubic grid; re-decorate as alternating (001) Zn and O layers
    new_sp  = ["Zn" if int(round(f[2] * 2)) % 2 == 0 else "O" for f in rs.frac_coords]
    layered = Structure(rs.lattice, new_sp, rs.frac_coords)
    species_test = {
        "composition_layered": layered.composition.reduced_formula,
        "PDD_EMD_rocksalt_vs_layered_A": emd_m(structure_pdd(rs), structure_pdd(layered), K_MAX),
        "StructureMatcher_type_match": bool(REFERENCES["type"].fit(rs, layered)),
    }
    print(f"\nSpecies test (rock salt vs layered decoration of the same sites): {species_test}")

# ── 6. Summary ───────────────────────────────────────────────────────────────
dedup_summary = {"threshold_interpretation": threshold_interpretation, "confirm_k": CONFIRM_K,
                 "confirm_ground_metric": METRIC_NAME[DEDUP_METRIC],
                 "lower_bound_max_violation_A": viol,
                 "k_grid": K_GRID, "tau_grid_A": TAU_GRID,
                 "representative_rule": "medoid; ties by source (COD>MP>OQMD>MATTERGEN), then ID",
                 "references": {r: str(m.as_dict()) for r, m in REFERENCES.items()},
                 "results": dedup_results, "species_test": species_test}
with open(dedup_dir / f"dedup_summary_{tag_m}.json", "w") as fh:
    json.dump(dedup_summary, fh, indent=2, default=str)

path = "A (AMD alone)" if dedup_results["AMD"]["type_false_merges"] == 0 else "B (AMD pre-filter + PDD)"
print(f"\nRecommended manuscript path: {path}")


In [ ]:
# %% [Cell 9b] Deduplication: groups, representatives, validation, sensitivity, species test
# ------------------------------------------------------------------
# Rule: DEDUP_LINKAGE linkage on the AMD distance (first DEDUP_K values, metric DEDUP_METRIC),
#       cut at DEDUP_TAU_A. With "complete" linkage every pair in a group is within DEDUP_TAU_A.
# Optional confirmation: PDD EMD at k = CONFIRM_K with the SAME ground metric, same threshold.
#       ||AMD_a − AMD_b|| ≤ EMD_k ≤ EMD_K (k ≤ K) for any norm, so the AMD step is lossless.
# Representative: group medoid (smallest summed distance to the other members);
#       ties → SOURCE_PRIORITY, then identifier.
# Reference: pymatgen StructureMatcher (species-aware) at two strictness levels.
# Requires Cells 1–9.
# ------------------------------------------------------------------
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import adjusted_rand_score
from pymatgen.analysis.structure_matcher import StructureMatcher

# ── SETTINGS ─────────────────────────────────────────────────────────────────
DEDUP_METRIC    = "euclidean"          # "euclidean" (as in the manuscript) or "chebyshev" (L∞)
DEDUP_K         = 5
DEDUP_TAU_A     = 0.1
DEDUP_LINKAGE   = "complete"           # "complete": every pair ≤ τ | "single": connected components
                                       # "ward": dendrogram cut (Euclidean only; NOT a pairwise bound)
CONFIRM_K       = max(17, K_TSNE_EFF)  # ≥ 17: first neighbour differing between ideal wurtzite and zinc blende
SHOW_WARD_CUT   = True                 # also report the Fig. 3(c) Ward dendrogram cut at τ (Euclidean only)
SENS_METRICS    = ("euclidean", "chebyshev")   # metrics included in the sensitivity sweep
SOURCE_PRIORITY = {"COD": 0, "MP": 1, "OQMD": 2, "MATTERGEN": 3, "other": 4}
TAU_GRID        = [0.0, 1e-3, 3e-3, 0.01, 0.02, 0.03, 0.05, 0.1, 0.2, 0.3]
K_GRID          = [k for k in [5, 10, 15, 17, 20, 25, 30, 50, 100] if k <= K_MAX]

METRIC_NAME = {"euclidean": "Euclidean (L2)", "chebyshev": "Chebyshev (L∞)"}
if DEDUP_METRIC not in METRIC_NAME:
    raise ValueError(f"DEDUP_METRIC must be one of {list(METRIC_NAME)}")
if DEDUP_LINKAGE == "ward" and DEDUP_METRIC != "euclidean":
    raise ValueError("Ward linkage requires DEDUP_METRIC = 'euclidean'")

REFERENCES = {
    "type": StructureMatcher(),                                                  # pymatgen defaults, scale=True
    "geom": StructureMatcher(ltol=0.01, stol=0.02, angle_tol=0.5, scale=False),  # geometric, ~τ-scale
}

dedup_dir = run_dir / "dedup"
dedup_dir.mkdir(exist_ok=True)
n_s = len(valid_names)
iu  = np.triu_indices(n_s, 1)
sg  = dict(zip(meta_df["file_name"], meta_df["space_group_symbol"]))
vpa = dict(zip(meta_df["file_name"], meta_df["volume_per_atom_A3"]))
tag_m = "L2" if DEDUP_METRIC == "euclidean" else "Linf"

# ── HELPERS ──────────────────────────────────────────────────────────────────
def amd_dist(k, metric=DEDUP_METRIC):
    """Square matrix of AMD distances using the first k AMD values."""
    X = np.array([amd_full[n][:k] for n in valid_names])
    return squareform(amd.AMD_pdist(X, metric=metric))

def emd_m(D1, D2, k, metric=DEDUP_METRIC):
    """PDD EMD truncated to k neighbours, with the given ground metric."""
    return float(amd.EMD(np.ascontiguousarray(D1[:, :k + 1]),
                         np.ascontiguousarray(D2[:, :k + 1]), metric=metric))

_pdd_cache = {}
def pdd_dist(k, metric=DEDUP_METRIC):
    """Square matrix of PDD EMDs at k (cached)."""
    if (k, metric) not in _pdd_cache:
        M = np.zeros((n_s, n_s))
        for i, j in zip(*iu):
            M[i, j] = M[j, i] = emd_m(pdd_store[valid_names[i]], pdd_store[valid_names[j]], k, metric)
        _pdd_cache[(k, metric)] = M
    return _pdd_cache[(k, metric)]

def prefix_dist(diff, metric=DEDUP_METRIC):
    """AMD distance using the first k components, for every k = 1…K."""
    return np.sqrt(np.cumsum(diff ** 2)) if metric == "euclidean" else np.maximum.accumulate(diff)

def group_labels(Dsq, tau, method=DEDUP_LINKAGE):
    return fcluster(linkage(squareform(Dsq, checks=False), method), tau, criterion="distance")

def coord_number(D, f=1.15):
    """Weighted mean number of neighbours within f × nearest-neighbour distance (from the PDD)."""
    d = D[:, 1:]
    return float(D[:, 0] @ (d <= f * d[:, [0]]).sum(axis=1))

cn = {n: coord_number(pdd_store[n]) for n in valid_names}

# ── 0. Precise interpretation of the threshold ───────────────────────────────
_sqrt_k = np.sqrt(DEDUP_K) if DEDUP_METRIC == "euclidean" else 1.0
threshold_interpretation = {
    "metric": METRIC_NAME[DEDUP_METRIC], "k": DEDUP_K, "tau_A": DEDUP_TAU_A, "linkage": DEDUP_LINKAGE,
    "max_difference_any_single_AMD_component_A": DEDUP_TAU_A,
    "max_rms_difference_over_k_components_A": DEDUP_TAU_A / np.sqrt(DEDUP_K) if DEDUP_METRIC == "euclidean" else DEDUP_TAU_A,
    "always_merged_if_every_atom_displaced_by_at_most_A": DEDUP_TAU_A / (2 * _sqrt_k),
    "pairwise_guarantee": {"complete": "every pair in a group is within tau",
                           "single": "groups are connected by chains of pairs within tau",
                           "ward": "none (variance-based merge cost)"}[DEDUP_LINKAGE],
}
print("Threshold interpretation:")
for key, v in threshold_interpretation.items():
    print(f"  {key}: {v:.4g}" if isinstance(v, float) else f"  {key}: {v}")

# ── 1. Independent reference: StructureMatcher (pairwise) ────────────────────
print("\nStructureMatcher references (species-aware, pairwise)…")
ref_match, ref_labels = {}, {}
for rname, matcher in REFERENCES.items():
    M = np.eye(n_s, dtype=bool)
    for i, j in zip(*iu):
        try:
            M[i, j] = M[j, i] = matcher.fit(all_structures[valid_names[i]], all_structures[valid_names[j]])
        except Exception as e:
            print(f"  {rname}: {_lbl[i]} vs {_lbl[j]} failed: {e}")
    ref_match[rname] = M
    n_comp, ref_labels[rname] = connected_components(M, directed=False)
    print(f"  {rname}: {int(M[iu].sum())} matching pairs → {n_comp} groups")

def evaluate(labels):
    same = labels[:, None] == labels[None, :]
    out = {"n_groups": int(len(set(labels)))}
    for rname, M in ref_match.items():
        tp = int((same & M)[iu].sum()); fp = int((same & ~M)[iu].sum()); fn = int((~same & M)[iu].sum())
        out.update({f"{rname}_false_merges": fp, f"{rname}_missed": fn,
                    f"{rname}_precision": tp / (tp + fp) if tp + fp else 1.0,
                    f"{rname}_recall":    tp / (tp + fn) if tp + fn else 1.0,
                    f"{rname}_ARI": float(adjusted_rand_score(ref_labels[rname], labels))})
    return out

# ── 2. Headline rules ────────────────────────────────────────────────────────
D_amd = amd_dist(DEDUP_K)
D_pdd = pdd_dist(CONFIRM_K)
viol  = float((D_amd - D_pdd)[iu].max())
tol_b = 2 * COLLAPSE_TOL * (np.sqrt(CONFIRM_K) if DEDUP_METRIC == "euclidean" else 1.0)
print(f"\nLower bound AMD(k={DEDUP_K}) ≤ PDD EMD(k={CONFIRM_K}), {METRIC_NAME[DEDUP_METRIC]}: "
      f"max violation {viol:.1e} Å ({'holds' if viol <= tol_b else 'CHECK'}; row-merging tolerance {tol_b:.1e} Å)")

RULES = {
    "AMD":     (D_amd, f"AMD {tag_m}, k={DEDUP_K}", DEDUP_LINKAGE),
    "AMD+PDD": (np.maximum(D_amd, D_pdd),
                f"AMD {tag_m} k={DEDUP_K} pre-filter → PDD EMD {tag_m} k={CONFIRM_K}", DEDUP_LINKAGE),
}
if SHOW_WARD_CUT and DEDUP_METRIC == "euclidean" and DEDUP_LINKAGE != "ward":
    RULES["Ward cut"] = (D_amd, f"Ward dendrogram (Fig. 3c), AMD L2 k={DEDUP_K}, cut at τ", "ward")

def group_tables(labels, Dsq):
    members, groups = [], []
    for g in sorted(set(labels)):
        idx    = np.where(labels == g)[0]
        others = np.where(labels != g)[0]
        sums   = Dsq[np.ix_(idx, idx)].sum(axis=1)
        order  = sorted(range(len(idx)), key=lambda t: (round(sums[t], 12),
                        SOURCE_PRIORITY[get_source(valid_names[idx[t]])], _lbl[idx[t]]))
        rep    = idx[order[0]]
        sub    = np.ix_(idx, idx)
        groups.append({
            "group": int(g), "representative": _lbl[rep], "n_members": len(idx),
            "members": "; ".join(_lbl[i] for i in idx),
            "space_groups": "; ".join(sorted({sg[valid_names[i]] for i in idx})),
            "max_intra_A": float(Dsq[sub].max()),
            "min_to_other_groups_A": float(Dsq[np.ix_(idx, others)].min()) if len(others) else np.nan,
            "all_pairs_match_type": bool(ref_match["type"][sub].all()),
            "all_pairs_match_geom": bool(ref_match["geom"][sub].all()),
        })
        for i in idx:
            nm = valid_names[i]
            members.append({"group": int(g), "structure": _lbl[i], "is_representative": i == rep,
                            "source": get_source(nm), "space_group": sg[nm],
                            "n_sites": all_structures[nm].num_sites, "coordination": round(cn[nm], 2),
                            "volume_per_atom_A3": vpa[nm], "distance_to_rep_A": float(Dsq[i, rep])})
    return pd.DataFrame(groups), pd.DataFrame(members)

def pair_audit(labels, Dsq):
    same, rows = labels[:, None] == labels[None, :], []
    for i, j in zip(*iu):
        a, b = valid_names[i], valid_names[j]
        for rname, M in ref_match.items():
            if same[i, j] == M[i, j]:
                continue
            pref  = prefix_dist(np.abs(amd_full[a] - amd_full[b]))
            k_sep = int(np.argmax(pref > DEDUP_TAU_A)) + 1 if (pref > DEDUP_TAU_A).any() else None
            if same[i, j]:
                kind = "false merge"
                if sg[a] == sg[b]:                    cat = "same space group"
                elif abs(cn[a] - cn[b]) < 0.5:        cat = "same coordination, different space group (stacking/ordering variant)"
                else:                                 cat = "different coordination (distinct polymorph)"
            else:
                kind = "missed duplicate"
                dv   = abs(vpa[a] - vpa[b]) / min(vpa[a], vpa[b])
                cat  = f"volume/atom differs by {100 * dv:.1f}%" + (" (scale, e.g. DFT vs experiment)" if dv > 0.01 else "")
            rows.append({"reference": rname, "type": kind, "a": _lbl[i], "b": _lbl[j],
                         "sg_a": sg[a], "sg_b": sg[b], "distance_A": float(Dsq[i, j]),
                         "AMD_first_k_exceeding_tau": k_sep, "category": cat})
    return pd.DataFrame(rows)

dedup_results = {}
for key, (Dsq, desc, method) in RULES.items():
    lab = group_labels(Dsq, DEDUP_TAU_A, method)
    gdf, mdf = group_tables(lab, Dsq)
    adf = pair_audit(lab, Dsq)
    ev  = evaluate(lab)
    split_close = int(((Dsq <= DEDUP_TAU_A) & (lab[:, None] != lab[None, :]))[iu].sum())
    tag = f"{key.replace(' ', '_').replace('+', '_')}_{tag_m}_{method}_k{DEDUP_K:03d}_tau{DEDUP_TAU_A:g}"
    gdf.to_csv(dedup_dir / f"groups_{tag}.csv", index=False)
    mdf.to_csv(dedup_dir / f"members_{tag}.csv", index=False)
    adf.to_csv(dedup_dir / f"pair_audit_{tag}.csv", index=False)
    dedup_results[key] = {"rule": desc, "linkage": method, **ev,
                          "pairs_within_tau_split": split_close,
                          "max_intra_group_distance_A": float(gdf["max_intra_A"].max()),
                          "representatives": gdf["representative"].tolist()}
    print(f"\n[{desc}, τ = {DEDUP_TAU_A} Å, {method} linkage]  {ev['n_groups']} groups from {n_s}")
    print(f"  representatives: {gdf['representative'].tolist()}")
    print(f"  largest within-group distance: {gdf['max_intra_A'].max():.4f} Å"
          + ("  ← exceeds τ (no pairwise guarantee)" if gdf["max_intra_A"].max() > DEDUP_TAU_A + 1e-12 else ""))
    for r in REFERENCES:
        print(f"  vs {r:4s}: false merges {ev[f'{r}_false_merges']:3d}, missed {ev[f'{r}_missed']:4d}, "
              f"precision {ev[f'{r}_precision']:.3f}, recall {ev[f'{r}_recall']:.3f}, ARI {ev[f'{r}_ARI']:.3f}")
    print(f"  pairs within τ split across groups: {split_close}")
    fm = adf[adf["type"] == "false merge"] if len(adf) else adf
    if len(fm):
        print("  False merges:")
        print(fm.drop_duplicates(["a", "b"]).to_string(index=False))

# ── 3. Polymorph check: wurtzite vs zinc blende ──────────────────────────────
for a, b in [("mp-2133.cif", "mp-1986.cif")]:
    if a in amd_full and b in amd_full:
        pref = prefix_dist(np.abs(amd_full[a] - amd_full[b]))
        print(f"\n{short_label(a)} ({sg[a]}) vs {short_label(b)} ({sg[b]}), {METRIC_NAME[DEDUP_METRIC]}: "
              f"AMD k=15 {pref[14]:.4f} | k=16 {pref[15]:.4f} | k=17 {pref[16]:.4f} Å; "
              f"PDD EMD k=17 {emd_m(pdd_store[a], pdd_store[b], 17):.4f} Å; "
              f"StructureMatcher(type) = {ref_match['type'][valid_names.index(a), valid_names.index(b)]}")

# ── 4. Threshold × k × metric sensitivity ────────────────────────────────────
metrics_sens = [m for m in SENS_METRICS if not (DEDUP_LINKAGE == "ward" and m != "euclidean")]
sens_rows = []
for metric in metrics_sens:
    for desc in ["AMD", "PDD"]:
        for k in K_GRID:
            Dk = amd_dist(k, metric) if desc == "AMD" else pdd_dist(k, metric)
            Zk = linkage(squareform(Dk, checks=False), DEDUP_LINKAGE)
            for tau in TAU_GRID:
                sens_rows.append({"metric": metric, "descriptor": desc, "k": k, "tau_A": tau,
                                  **evaluate(fcluster(Zk, tau, criterion="distance"))})
sens_df = pd.DataFrame(sens_rows)
sens_df.to_csv(dedup_dir / "threshold_sensitivity.csv", index=False)

S = sens_df[sens_df.metric == DEDUP_METRIC]
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
for k in [5, 15, 30, 100]:
    s = S[(S.descriptor == "AMD") & (S.k == k)]
    axs[0].plot(s.tau_A, s.n_groups, marker="o", ms=3, label=f"AMD, k={k}")
s = S[(S.descriptor == "PDD") & (S.k == CONFIRM_K)]
axs[0].plot(s.tau_A, s.n_groups, "k--", marker="s", ms=3, label=f"PDD EMD, k={CONFIRM_K}")
for m_alt in [m for m in metrics_sens if m != DEDUP_METRIC]:
    s = sens_df[(sens_df.metric == m_alt) & (sens_df.descriptor == "AMD") & (sens_df.k == DEDUP_K)]
    axs[0].plot(s.tau_A, s.n_groups, color="grey", ls=":", marker="^", ms=3,
                label=f"AMD, k={DEDUP_K}, {METRIC_NAME[m_alt]}")
for desc, k, col in [("AMD", DEDUP_K, "#1565C0"), ("PDD", CONFIRM_K, "black")]:
    s = S[(S.descriptor == desc) & (S.k == k)]
    for r, ls in [("geom", "-"), ("type", ":")]:
        axs[1].plot(s.tau_A, s[f"{r}_false_merges"], ls, color=col, marker="o", ms=3, label=f"{desc} k={k}, {r}")
        axs[2].plot(s.tau_A, s[f"{r}_missed"],       ls, color=col, marker="o", ms=3, label=f"{desc} k={k}, {r}")
for ax, yl in zip(axs, ["Number of groups", "False merges (pairs)", "Missed duplicates (pairs)"]):
    ax.set_xscale("symlog", linthresh=1e-3)
    ax.axvline(DEDUP_TAU_A, color="grey", ls="--", lw=1)
    ax.set_xlabel(f"Threshold τ [Å] ({METRIC_NAME[DEDUP_METRIC]})")
    ax.set_ylabel(yl)
    ax.grid(True, ls="--", alpha=0.5)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(dedup_dir / f"figS_dedup_sensitivity_{tag_m}.png", dpi=300, bbox_inches="tight")
fig.savefig(dedup_dir / f"figS_dedup_sensitivity_{tag_m}.pdf", bbox_inches="tight")
plt.show()

# ── 5. Species handling: same positions, different chemistry ─────────────────
species_test = {}
if "mp-2229.cif" in all_structures:
    rs = SpacegroupAnalyzer(all_structures["mp-2229.cif"]).get_conventional_standard_structure()
    # Rock-salt sites lie on a simple-cubic grid; re-decorate as alternating (001) Zn and O layers
    new_sp  = ["Zn" if int(round(f[2] * 2)) % 2 == 0 else "O" for f in rs.frac_coords]
    layered = Structure(rs.lattice, new_sp, rs.frac_coords)
    species_test = {
        "composition_layered": layered.composition.reduced_formula,
        "PDD_EMD_rocksalt_vs_layered_A": emd_m(structure_pdd(rs), structure_pdd(layered), K_MAX),
        "StructureMatcher_type_match": bool(REFERENCES["type"].fit(rs, layered)),
    }
    print(f"\nSpecies test (rock salt vs layered decoration of the same sites): {species_test}")

# ── 6. Summary ───────────────────────────────────────────────────────────────
dedup_summary = {"threshold_interpretation": threshold_interpretation, "confirm_k": CONFIRM_K,
                 "confirm_ground_metric": METRIC_NAME[DEDUP_METRIC],
                 "lower_bound_max_violation_A": viol,
                 "representative_rule": "medoid; ties by source (COD>MP>OQMD>MATTERGEN), then ID",
                 "references": {r: str(m.as_dict()) for r, m in REFERENCES.items()},
                 "results": dedup_results, "species_test": species_test}
with open(dedup_dir / f"dedup_summary_{tag_m}.json", "w") as fh:
    json.dump(dedup_summary, fh, indent=2, default=str)

path = "A (AMD alone)" if dedup_results["AMD"]["type_false_merges"] == 0 else "B (AMD pre-filter + PDD)"
print(f"\nRecommended manuscript path: {path}")


In [ ]:
# # %% [Cell 9b] Deduplication: groups, representatives, validation, sensitivity, species test
# # ------------------------------------------------------------------
# # Rule: complete linkage on the L∞ AMD distance (first DEDUP_K values), cut at DEDUP_TAU_A
# #       → every pair in a group has every AMD component within DEDUP_TAU_A.
# # Optional confirmation: PDD EMD (L∞ ground metric) at k = CONFIRM_K, same threshold.
# #       ||AMD_a − AMD_b||∞ ≤ EMD_k ≤ EMD_K (k ≤ K), so the AMD step is a lossless pre-filter.
# # Representative: group medoid (smallest summed distance to the other members);
# #       ties → SOURCE_PRIORITY, then identifier.
# # Reference: pymatgen StructureMatcher (species-aware), two strictness levels.
# # ------------------------------------------------------------------
# from scipy.cluster.hierarchy import fcluster, linkage
# from scipy.sparse.csgraph import connected_components
# from sklearn.metrics import adjusted_rand_score
# from pymatgen.analysis.structure_matcher import StructureMatcher

# DEDUP_K         = 5
# DEDUP_TAU_A     = 0.03
# DEDUP_METRIC    = "euclidean"           # "euclidean" = AMD L∞, "cityblock" = AMD L1
# DEDUP_LINKAGE   = "complete"            # "single" = connected components (allows chaining)
# CONFIRM_K       = max(17, K_TSNE_EFF)   # ≥ 17: first neighbour differing between ideal wurtzite and zinc blende
# SOURCE_PRIORITY = {"COD": 0, "MP": 1, "OQMD": 2, "MATTERGEN": 3, "other": 4}
# TAU_GRID        = [0.0, 1e-3, 3e-3, 0.01, 0.02, 0.03, 0.05, 0.1, 0.2, 0.3]
# K_GRID          = [k for k in [5, 10, 15, 17, 20, 25, 30, 50, 100] if k <= K_MAX]

# REFERENCES = {
#     "type": StructureMatcher(),                                                  # pymatgen defaults, scale=True
#     "geom": StructureMatcher(ltol=0.01, stol=0.02, angle_tol=0.5, scale=False),  # geometric, ~τ-scale
# }

# dedup_dir = run_dir / "dedup"
# dedup_dir.mkdir(exist_ok=True)
# n_s = len(valid_names)
# iu  = np.triu_indices(n_s, 1)
# sg  = dict(zip(meta_df["file_name"], meta_df["space_group_symbol"]))
# vpa = dict(zip(meta_df["file_name"], meta_df["volume_per_atom_A3"]))

# def amd_dist(k):
#     X = np.array([amd_full[n][:k] for n in valid_names])
#     return squareform(amd.AMD_pdist(X, metric=DEDUP_METRIC))

# def group_labels(Dsq, tau, method=DEDUP_LINKAGE):
#     return fcluster(linkage(squareform(Dsq, checks=False), method), tau, criterion="distance")

# def coord_number(D, f=1.15):
#     """Weighted mean number of neighbours within f × nearest-neighbour distance (from the PDD)."""
#     d = D[:, 1:]
#     return float(D[:, 0] @ (d <= f * d[:, [0]]).sum(axis=1))

# cn = {n: coord_number(pdd_store[n]) for n in valid_names}

# # ── 1. Independent reference: StructureMatcher (pairwise) ────────────────────
# print("StructureMatcher references (species-aware, pairwise)…")
# ref_match, ref_labels = {}, {}
# for rname, matcher in REFERENCES.items():
#     M = np.eye(n_s, dtype=bool)
#     for i, j in zip(*iu):
#         try:
#             M[i, j] = M[j, i] = matcher.fit(all_structures[valid_names[i]], all_structures[valid_names[j]])
#         except Exception as e:
#             print(f"  {rname}: {_lbl[i]} vs {_lbl[j]} failed: {e}")
#     ref_match[rname] = M
#     n_comp, ref_labels[rname] = connected_components(M, directed=False)
#     print(f"  {rname}: {int(M[iu].sum())} matching pairs → {n_comp} groups")

# def evaluate(labels):
#     same = labels[:, None] == labels[None, :]
#     out = {"n_groups": int(len(set(labels)))}
#     for rname, M in ref_match.items():
#         tp = int((same & M)[iu].sum()); fp = int((same & ~M)[iu].sum()); fn = int((~same & M)[iu].sum())
#         out.update({f"{rname}_false_merges": fp, f"{rname}_missed": fn,
#                     f"{rname}_precision": tp / (tp + fp) if tp + fp else 1.0,
#                     f"{rname}_recall":    tp / (tp + fn) if tp + fn else 1.0,
#                     f"{rname}_ARI": float(adjusted_rand_score(ref_labels[rname], labels))})
#     return out

# # ── 2. Headline rules ────────────────────────────────────────────────────────
# D_amd = amd_dist(DEDUP_K)
# D_pdd = pairwise_emd(valid_names, pdd_store, CONFIRM_K)
# viol  = float((D_amd - D_pdd)[iu].max())
# print(f"\nLower bound AMD(k={DEDUP_K}) ≤ PDD EMD(k={CONFIRM_K}): max violation {viol:.1e} Å "
#       f"({'holds' if viol <= 2 * COLLAPSE_TOL else 'CHECK'}; tolerance 2 × collapse_tol)")

# RULES = {"AMD": (D_amd, f"AMD L∞, k={DEDUP_K}"),
#          "AMD+PDD": (np.maximum(D_amd, D_pdd), f"AMD k={DEDUP_K} pre-filter → PDD EMD k={CONFIRM_K}")}

# def group_tables(labels, Dsq):
#     members, groups = [], []
#     for g in sorted(set(labels)):
#         idx    = np.where(labels == g)[0]
#         others = np.where(labels != g)[0]
#         sums   = Dsq[np.ix_(idx, idx)].sum(axis=1)
#         order  = sorted(range(len(idx)), key=lambda t: (round(sums[t], 12),
#                         SOURCE_PRIORITY[get_source(valid_names[idx[t]])], _lbl[idx[t]]))
#         rep    = idx[order[0]]
#         sub    = np.ix_(idx, idx)
#         groups.append({
#             "group": int(g), "representative": _lbl[rep], "n_members": len(idx),
#             "members": "; ".join(_lbl[i] for i in idx),
#             "space_groups": "; ".join(sorted({sg[valid_names[i]] for i in idx})),
#             "max_intra_A": float(Dsq[sub].max()),
#             "min_to_other_groups_A": float(Dsq[np.ix_(idx, others)].min()) if len(others) else np.nan,
#             "all_pairs_match_type": bool(ref_match["type"][sub].all()),
#             "all_pairs_match_geom": bool(ref_match["geom"][sub].all()),
#         })
#         for i in idx:
#             nm = valid_names[i]
#             members.append({"group": int(g), "structure": _lbl[i], "is_representative": i == rep,
#                             "source": get_source(nm), "space_group": sg[nm],
#                             "n_sites": all_structures[nm].num_sites, "coordination": round(cn[nm], 2),
#                             "volume_per_atom_A3": vpa[nm], "distance_to_rep_A": float(Dsq[i, rep])})
#     return pd.DataFrame(groups), pd.DataFrame(members)

# def pair_audit(labels, Dsq):
#     same, rows = labels[:, None] == labels[None, :], []
#     for i, j in zip(*iu):
#         a, b = valid_names[i], valid_names[j]
#         for rname, M in ref_match.items():
#             if same[i, j] == M[i, j]:
#                 continue
#             diff  = np.abs(amd_full[a] - amd_full[b])
#             k_sep = int(np.argmax(diff > DEDUP_TAU_A)) + 1 if (diff > DEDUP_TAU_A).any() else None
#             if same[i, j]:
#                 kind = "false merge"
#                 if sg[a] == sg[b]:                    cat = "same space group"
#                 elif abs(cn[a] - cn[b]) < 0.5:        cat = "same coordination, different space group (stacking/ordering variant)"
#                 else:                                 cat = "different coordination (distinct polymorph)"
#             else:
#                 kind = "missed duplicate"
#                 dv   = abs(vpa[a] - vpa[b]) / min(vpa[a], vpa[b])
#                 cat  = f"volume/atom differs by {100 * dv:.1f}%" + (" (scale, e.g. DFT vs experiment)" if dv > 0.01 else "")
#             rows.append({"reference": rname, "type": kind, "a": _lbl[i], "b": _lbl[j],
#                          "sg_a": sg[a], "sg_b": sg[b], "distance_A": float(Dsq[i, j]),
#                          "AMD_first_k_exceeding_tau": k_sep, "category": cat})
#     return pd.DataFrame(rows)

# dedup_results = {}
# for key, (Dsq, desc) in RULES.items():
#     lab = group_labels(Dsq, DEDUP_TAU_A)
#     gdf, mdf = group_tables(lab, Dsq)
#     adf = pair_audit(lab, Dsq)
#     ev  = evaluate(lab)
#     split_close = int(((Dsq <= DEDUP_TAU_A) & (lab[:, None] != lab[None, :]))[iu].sum())
#     tag = f"{key}_k{DEDUP_K:03d}_tau{DEDUP_TAU_A:g}"
#     gdf.to_csv(dedup_dir / f"groups_{tag}.csv", index=False)
#     mdf.to_csv(dedup_dir / f"members_{tag}.csv", index=False)
#     adf.to_csv(dedup_dir / f"pair_audit_{tag}.csv", index=False)
#     dedup_results[key] = {"rule": desc, **ev, "pairs_within_tau_split": split_close,
#                           "representatives": gdf["representative"].tolist()}
#     print(f"\n[{desc}, τ = {DEDUP_TAU_A} Å, {DEDUP_LINKAGE} linkage]  {ev['n_groups']} groups from {n_s}")
#     print(f"  representatives: {gdf['representative'].tolist()}")
#     for r in REFERENCES:
#         print(f"  vs {r:4s}: false merges {ev[f'{r}_false_merges']:3d}, missed {ev[f'{r}_missed']:4d}, "
#               f"precision {ev[f'{r}_precision']:.3f}, recall {ev[f'{r}_recall']:.3f}, ARI {ev[f'{r}_ARI']:.3f}")
#     print(f"  pairs within τ split across groups: {split_close}")
#     if len(adf):
#         fm = adf[adf["type"] == "false merge"]
#         if len(fm):
#             print("  False merges:")
#             print(fm.drop_duplicates(["a", "b"]).to_string(index=False))

# # ── 3. Polymorph / stacking-variant check: wurtzite vs zinc blende ───────────
# for a, b in [("mp-2133.cif", "mp-1986.cif")]:
#     if a in amd_full and b in amd_full:
#         diff = np.abs(amd_full[a] - amd_full[b])
#         print(f"\n{short_label(a)} ({sg[a]}) vs {short_label(b)} ({sg[b]}): AMD L∞ "
#               f"k=15 {diff[:15].max():.4f} | k=16 {diff[:16].max():.4f} | k=17 {diff[:17].max():.4f} Å; "
#               f"PDD EMD k=17 {emd(pdd_store[a], pdd_store[b], 17):.4f} Å; "
#               f"StructureMatcher(type) = {ref_match['type'][valid_names.index(a), valid_names.index(b)]}")

# # ── 4. Threshold × k sensitivity ─────────────────────────────────────────────
# sens_rows = []
# for desc in ["AMD", "PDD"]:
#     for k in K_GRID:
#         Dk = amd_dist(k) if desc == "AMD" else pairwise_emd(valid_names, pdd_store, k)
#         Zk = linkage(squareform(Dk, checks=False), DEDUP_LINKAGE)
#         for tau in TAU_GRID:
#             sens_rows.append({"descriptor": desc, "k": k, "tau_A": tau,
#                               **evaluate(fcluster(Zk, tau, criterion="distance"))})
# sens_df = pd.DataFrame(sens_rows)
# sens_df.to_csv(dedup_dir / "threshold_sensitivity.csv", index=False)

# fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
# for k in [5, 15, 30, 100]:
#     s = sens_df[(sens_df.descriptor == "AMD") & (sens_df.k == k)]
#     axs[0].plot(s.tau_A, s.n_groups, marker="o", ms=3, label=f"AMD, k={k}")
# s = sens_df[(sens_df.descriptor == "PDD") & (sens_df.k == CONFIRM_K)]
# axs[0].plot(s.tau_A, s.n_groups, "k--", marker="s", ms=3, label=f"PDD EMD, k={CONFIRM_K}")
# for (desc, k, col) in [("AMD", DEDUP_K, "#1565C0"), ("PDD", CONFIRM_K, "black")]:
#     s = sens_df[(sens_df.descriptor == desc) & (sens_df.k == k)]
#     for r, ls in [("geom", "-"), ("type", ":")]:
#         axs[1].plot(s.tau_A, s[f"{r}_false_merges"], ls, color=col, marker="o", ms=3, label=f"{desc} k={k}, {r}")
#         axs[2].plot(s.tau_A, s[f"{r}_missed"],       ls, color=col, marker="o", ms=3, label=f"{desc} k={k}, {r}")
# for ax, yl in zip(axs, ["Number of groups", "False merges (pairs)", "Missed duplicates (pairs)"]):
#     ax.set_xscale("symlog", linthresh=1e-3)
#     ax.axvline(DEDUP_TAU_A, color="grey", ls="--", lw=1)
#     ax.set_xlabel("Threshold τ [Å]")
#     ax.set_ylabel(yl)
#     ax.grid(True, ls="--", alpha=0.5)
#     ax.legend(fontsize=8)
# fig.tight_layout()
# fig.savefig(dedup_dir / "figS_dedup_sensitivity.png", dpi=300, bbox_inches="tight")
# fig.savefig(dedup_dir / "figS_dedup_sensitivity.pdf", bbox_inches="tight")
# plt.show()

# # ── 5. Species handling: same positions, different chemistry ─────────────────
# species_test = {}
# if "mp-2229.cif" in all_structures:
#     rs  = SpacegroupAnalyzer(all_structures["mp-2229.cif"]).get_conventional_standard_structure()
#     # All rock-salt sites lie on a simple-cubic grid; re-decorate as alternating (001) Zn and O layers
#     new_sp  = ["Zn" if int(round(f[2] * 2)) % 2 == 0 else "O" for f in rs.frac_coords]
#     layered = Structure(rs.lattice, new_sp, rs.frac_coords)
#     species_test = {
#         "composition_layered": layered.composition.reduced_formula,
#         "PDD_EMD_rocksalt_vs_layered_A": emd(structure_pdd(rs), structure_pdd(layered), K_MAX),
#         "StructureMatcher_type_match": bool(REFERENCES["type"].fit(rs, layered)),
#     }
#     print(f"\nSpecies test (rock salt vs layered decoration of the same sites): {species_test}")

# # ── 6. Summary for the rebuttal ──────────────────────────────────────────────
# dedup_summary = {"k": DEDUP_K, "tau_A": DEDUP_TAU_A, "metric": "L∞", "linkage": DEDUP_LINKAGE,
#                  "confirm_k": CONFIRM_K, "representative_rule": "medoid; ties by source (COD>MP>OQMD>MATTERGEN), then ID",
#                  "references": {r: str(m.as_dict()) for r, m in REFERENCES.items()},
#                  "results": dedup_results, "species_test": species_test}
# with open(dedup_dir / "dedup_summary.json", "w") as fh:
#     json.dump(dedup_summary, fh, indent=2, default=str)

# path = "A (AMD alone)" if dedup_results["AMD"]["type_false_merges"] == 0 else "B (AMD pre-filter + PDD)"
# print(f"\nRecommended manuscript path: {path}")


In [ ]:
# %% Deduplication at the manuscript threshold
from scipy.cluster.hierarchy import fcluster
DEDUP_THRESHOLD_A = 0.03
X_dd = np.array([amd_full[n][:K_BASELINE] for n in valid_names])
d_dd = amd.AMD_pdist(X_dd, metric=AMD_PDIST_METRIC)
# single linkage = structures joined by chains of pairs closer than the threshold
groups = fcluster(linkage(d_dd, "single"), DEDUP_THRESHOLD_A, criterion="distance")
print(f"Unique structures at {DEDUP_THRESHOLD_A} Å (k={K_BASELINE}): {len(set(groups))} / {len(valid_names)}")
for t in [0.005, 0.01, 0.02, 0.03, 0.05, 0.1]:   # sensitivity to the threshold
    print(f"  {t:5.3f} Å → {len(set(fcluster(linkage(d_dd, 'single'), t, criterion='distance')))}")

In [ ]:
# %% [Cell 10] 3D structure viewer
# Paste your existing viewer cell here UNCHANGED. It computes no descriptors, so the
# coordinate fix does not affect it. Render it only after Cell 9 has confirmed
# structures_similar / structures_dissimilar. Unused imports (CifParser, tempfile,
# ase, JMOL_COLORS) can be removed.


In [ ]:
# %% [Cell 11] OPTIONAL — compare archived (submitted) PDDs with the faulty and corrected paths
LEGACY_RUN_DIR  = None   # e.g. RESULTS_BASE_DIR / "run_2026XXXX_XXXXXX"  (run behind the submitted figure)
LEGACY_FRAC_TOL = 1e-4   # the submitted run used pymatgen's default rounding

if LEGACY_RUN_DIR is not None:
    with open(Path(LEGACY_RUN_DIR) / "pdd_store.pkl", "rb") as fh:
        legacy_store = pickle.load(fh)
    rows = []
    for name in sorted(set(legacy_store) & set(pdd_store)):
        s_old = parse_cif(cif_paths[name], LEGACY_FRAC_TOL)[0]
        D_old = np.asarray(legacy_store[name], dtype=float)
        D_rep = compute_pdd(legacy_fractional_periodicset(s_old), K_MAX)
        rows.append({"name": short_label(name),
                     "EMD(archived, faulty path reproduced) [Å]": emd(D_old, D_rep, K_MAX),
                     "EMD(archived, corrected) [Å]":             emd(D_old, pdd_store[name], K_MAX)})
    cmp_df = pd.DataFrame(rows)
    cmp_df.to_csv(run_dir / "legacy_vs_corrected.csv", index=False)
    print(cmp_df.describe())


In [ ]:
# %% [Cell 12] All numbers needed for the rebuttal / Methods → rebuttal_numbers.json
def _py(x):
    if x is None or (not isinstance(x, (list, dict, str)) and pd.isna(x)):
        return None
    return x.item() if hasattr(x, "item") else x

_sum = summary_df.set_index("id")
numbers = {
    "n_structures":        len(valid_names),
    "COLLAPSE_TOL_A":      COLLAPSE_TOL,
    "FRAC_TOLERANCE":      FRAC_TOLERANCE,
    "AMD_METRIC":          AMD_METRIC,
    "AMD_PDIST_METRIC":     AMD_PDIST_METRIC,
    "LINKAGE_METHOD":      LINKAGE_METHOD,
    "global_max_emd_kmax_A": global_max_emd,
    "audit": {tid: {"label": _sum.loc[tid, "test"], "n": _py(_sum.loc[tid, "n"]),
                    "n_passed": _py(_sum.loc[tid, "n_passed"]), "n_failed": _py(_sum.loc[tid, "n_failed"]),
                    "n_nonzero": _py(_sum.loc[tid, "n_nonzero"]),
                    "min": _py(_sum.loc[tid, "min_value"]), "max": _py(_sum.loc[tid, "max_value"])}
              for tid in _sum.index},
    "artefact_jump_at_n_sites": f"{n_jump_match}/{len(jump_df)}",
    "periodic_models_k_reliable": {short_label(n): k_rel[n] for n in model_names},
    "K_TSNE_EFF":          K_TSNE_EFF,
    "tsne_perplexity":     TSNE_PERPLEXITY_EFF,
    "tsne_kl_divergence":  float(tsne.kl_divergence_),
    "rounding_sensitivity": sensitivity,
}

if PLOT_K_SWEEP_SUMMARY:
    numbers["K_TSNE_MAIN_FIGURE"] = K_TSNE_MAIN_FIG
else:
    numbers["K_TSNE_MAIN_FIGURE"] = None

with open(run_dir / "rebuttal_numbers.json", "w") as fh:
    json.dump(numbers, fh, indent=2, default=_py)

a = numbers["audit"]
print(f"""Key values for the audit
  Structures                         : {numbers['n_structures']}
  Parser cross-check (A6a)           : {a['A6a']['n_passed']}/{a['A6a']['n']}, max {a['A6a']['max']:.2e} Å
  Rounding effect (A6b)              : {a['A6b']['n_nonzero']} structures, max {a['A6b']['max']:.2e} Å
  Primitive vs conventional (C3)     : max {a['C3']['max']:.2e} Å
  Supercell (C1) / sheared basis (C2): max {a['C1']['max']:.2e} / {a['C2']['max']:.2e} Å
  Analytic AMD_1 (B1)                : max {a['B1']['max']:.2e} Å ({a['B1']['n']} structures)
  Min nearest-neighbour (A4)         : {a['A4']['min']:.3f} Å
  Artefact size (E1)                 : {a['E1']['min']:.3f}–{a['E1']['max']:.3f} Å
  Artefact max d_(n-1) (E2)          : {a['E2']['max']:.3f} (fractional units)
  Artefact jump d_n (E3)             : {a['E3']['min']:.2f}–{a['E3']['max']:.2f} Å
  Flattened vs EMD under perturbation: {a['D2']['max']:.3e} vs {a['D3']['max']:.3e} Å
  k_reliable (periodic models)       : {numbers['periodic_models_k_reliable']}
  t-SNE k                            : {K_TSNE_EFF}
  Rounding sensitivity               : leaf order identical = {sensitivity['dendrogram_leaf_order_identical']}, ARI = {sensitivity['flat_clusters_ARI']:.4f}
""")
